### ShapeNet: Geometry-Guided Neural Networks for Microscopy Image Analysis


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_25_12112026/shapeNet_data_.ipynb)

**Aidan Swanger**
With outputs, file was too big for github, rerun the notebook for outputs, or I have another version

## 1. Setup

In [ ]:
%pip install -q pyTEMlib ipympl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done


In [ ]:
! gdown --folder "https://drive.google.com/drive/folders/1h2v6YRPG7cwi7kuMPPPdwV0P0PYu49oH?usp=sharing"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import re
import numpy as np
import pyTEMlib.file_tools as ft
import matplotlib.pyplot as plt

def natural_sort_key(path: Path):
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", path.name)
    ]


def list_emd_files(folder: Path) -> list[Path]:
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".emd"],
        key=natural_sort_key,
    )

def load_haadf(file: Path) -> np.ndarray:
    """Load HAADF if available, otherwise Channel_000 or the first 2D dataset."""
    datasets = ft.open_file(str(file))

    if not isinstance(datasets, dict):
        arr = np.squeeze(np.asarray(datasets))
        if arr.ndim != 2:
            raise ValueError(f"No 2D image found in {Path(file).name}")
        return arr

    for key, dset in datasets.items():
        title = str(getattr(dset, "title", ""))
        name = str(getattr(dset, "name", ""))
        if "HAADF" in f"{key} {title} {name}".upper():
            arr = np.squeeze(np.asarray(dset))
            if arr.ndim == 2:
                return arr

    if "Channel_000" in datasets:
        arr = np.squeeze(np.asarray(datasets["Channel_000"]))
        if arr.ndim == 2:
            return arr

    for dset in datasets.values():
        try:
            arr = np.squeeze(np.asarray(dset))
            if arr.ndim == 2:
                return arr
        except Exception:
            pass

    raise ValueError(f"No 2D dataset found in {Path(file).name}")

## 2. Load the images

In [ ]:
SISB_DIR = "day_25 data/SiSb/"
YBCO_DIR = "day_25 data/YBCO/"

In [ ]:
all_sisb_files = list_emd_files(SISB_DIR)
all_ybco_files = list_emd_files(YBCO_DIR)

N_SISB = 12
sisb_files = all_sisb_files[:N_SISB]
ybco_files = all_ybco_files[:]
sisb_images = [
    load_haadf(file)
    for file in sisb_files
]

ybco_images = [
    load_haadf(file)
    for file in ybco_files
]

In [ ]:
plt.imshow(sisb_images[4], cmap = "grey")
plt.axis("off")

In [ ]:
plt.imshow(ybco_images[4], cmap = "grey")
plt.axis("off")

## 3. <font color = "red">**Hackathon Instructions**</font>

In this hackathon, we will explore **lightweight, object-specific neural networks** designed around the geometrical features and shapes of objects in microscopy images.

### Choose one of the following challenges

**Challenge 1 — Boundary Detection (SiSb)**

- Identify the boundary separating crystalline and amorphous regions.
- Explore geometric features that distinguish these regions, such as local periodicity, texture, and structural order.
- Develop or propose a lightweight neural network to detect the boundary.
- Visualize the detected boundary overlaid on the original image.

**Challenge 2 — Atom Finding (YBCO)**

- Identify atomic columns in the STEM images.
- Explore shape-based features such as atomic-column intensity profiles, symmetry, size, and spatial arrangement.
- Develop or propose a lightweight neural network for atomic-column detection.
- Return the detected atomic coordinates and visualize them on the original image.

### What you should do

1. **Explore the dataset:** Inspect the images and identify the relevant structural features.
2. **Define the problem:** Determine which geometrical features are most useful for detecting the target objects.
3. **Explore network design:** Consider how these features could be incorporated into a small, object-specific neural network.
4. **Test your approach:** If possible, implement a preliminary method and visualize its predictions.
5. **Discuss your findings:** Explain what works, what does not, and the main limitations of your approach.

You are also encouraged to **use your own experimental images and scientific problems** to explore whether the same approach can be generalized to other materials and imaging modalities.

**The goal is exploratory:** A fully trained or optimized model is not required. Focus on the connection between object geometry, network design, and detection performance.

# **My Work:**

### Visualize All YBCO Images

In [ ]:
for file, img in zip(all_ybco_files, ybco_images):
  plt.figure(figsize = (5, 5))
  plt.imshow(img, cmap = "gray")
  plt.title(f"{file.name} {img.shape}")
  plt.axis("off")
  plt.show()

# ShapeNet-YBCO: geometry-guided atomic-column finding in YBCO HAADF-STEM images

**Summary:** V1 finds the heavy columns almost perfectly on synthetic data. V2a finds Cu from its brightness: it marks only 0.2% of sites as empty Cu sites when Cu is removed, and on the 14 real images, its Cu counts match the crystal structure (Cu:heavy 1.007, Cu2:Cu1 1.999). The residual approach (V2b) places 12% of its Cu off-site on the real images.

## What I analyzed
* 18 YBCO HAADF-STEM images (Velox `.emd`, 2048×2048 px, 200 kV, 0.119–0.477 Å/px). **14 images chosen and used.** Scan image numbers 1717, 1338, 1520 and 1734 are dropped: some of their sampled regions do not fold into a clean YBCO unit cell form (Section A). All images are side views along [100]/[010]. Each cell shows 3 heavy columns (Ba, Y, Ba pattern) and 3 Cu columns (Cu1, Cu2, Cu2 pattern). Y/Ba brightness is 0.87, so Y and Ba form one heavy atomic column class; Cu/heavy brightness is 0.52.
* **Goal:** Find the heavy columns first so that the user can visibly check the model. Then, find the faint Cu columns (the real test because these columns are difficult to discern), and overlay both on the original HAADF-STEM images.  This is primarily to test if this shapeNet approach is viable.

## Filters and network (about 16,200 parameters, 78 of them being analytic shape parameters)
* **Blob bank:** 8 near-circular, elliptical Gaussians with learnable width, axis ratio (0.75–1), and angle.
* **Chain bank ("columns of blobs"):** 6 operators that reinforce blobs lying in lattice rows with learnable spacing and direction, started at the YBCO row spacings.
* **Analytic U-Net:** Gaussian, oriented-derivative, and LoG kernels with learnable widths, plus 1×1 channel mixing (13×13 windows, reflect padding).
* **Two outputs:** A class map and the distance to the nearest column center. Columns are the peaks of **P(class) × (1 − distance)**.

| Version | Input | Finds |
|---|---|---|
| V1 | HAADF | heavy columns |
| V2a | HAADF | heavy + Cu (the independent Cu test) |
| V2b | HAADF minus V1's fitted heavy columns | Cu |

## How I trained it
* **Synthetic data (rendered then augmented):** Images are rendered from scratch from the YBa₂Cu₃O₇ structure with measured column widths, brightness ratios, and noise (weak oxygen columns are labelled background for now). They then go through `normalize_image` and `augment_pair`, which are also used in my AtomAI U-Net notebook. No real image or AtomAI/Atomap label is used in training, so all 14 real images are unseen.
* **Cu vacancies:** 0–50% of the Cu columns are removed in each crop (10% of crops have none) and labeled background, so the model can only score on Cu where Cu brightness is present.
* **3,000 / 300 / 300 training / validation / test** crops of 256×256 px. Adam (learning rate 3×10⁻³), batch 8, 15 epochs, best-validation checkpoint. Detection thresholds are chosen on validation then frozen at that threshold.

## Results
| | V1 | V2a | V2b |
|---|---|---|---|
| Synthetic test F1, heavy | 0.9997 | 0.9998 | — |
| Synthetic test F1, Cu | — | 0.9969 | 0.9878 |
| Empty Cu sites marked as Cu, all Cu removed (ideal 0) | — | 0.2% (19 of 8,840) | 0.01% (1 of 8,840) |
| Empty Cu sites marked / Cu recall, half removed | — | 0.02% / 99.9% | 0.02% / 100.0% |
| Real images, Cu : heavy (crystal value 1), median (range) | — | 1.007 (0.996–1.042) | 1.018 (1.000–1.039) |
| Real images, Cu2 : Cu1 (crystal value 2), median (range) | — | 1.999 (1.873–2.339) | 2.023 (1.884–2.665) |
| Real images, Cu off-site (site ratio < 0.6), median | — | 1.45% | 12.2% (up to 59% in 1333) |

**Detection limit:** V2a Cu F1 stays at or above 0.988 down to Cu/heavy = 0.25 (0.941 at 0.15); V2b stays at or above 0.970 down to 0.25 (0.828 at 0.15). Real images: median 0.52 (IQR 0.48–0.53).

**What works:**
* **Heavy columns:** F1 ≈ 1.000 on synthetic data (**position error 0.35 px ≈ 0.09 Å**) with consistent counts on all 14 real images.
* **V2a Cu uses brightness, not lattice position:** it marks 0.2% of empty Cu sites when all Cu is removed and finds 99.9% of the Cu that remains when half is removed.  This confirms that the model can distinguish between when no Cu column exists and when a Cu column exists.
* **V2a matches the crystal structure on the real images:** Cu:heavy ≈ 1 and Cu2:Cu1 ≈ 2 in all 14 images, and the Cu site histogram shows two peaks at the Cu2 (approx 0.70) and Cu1 (approx 0.80) positions with almost nothing at the edge midpoint (0.5).
* **The learned shapes stay physical:** blob widths (1.0–4.6 px) bracket the measured column width (2.75 px), and chain spacings stay within 2 px of the crystal spacings they started from.
* **Training is light:** about 21 min per model on a L4 GPU (15 epochs) (30 epochs was too time costly).

**What doesn't work and limitations:**
* **V2b on real images:** 12.2% of its Cu sits off-site (median), 59% in 1333 (where Cu2:Cu1 = 2.67), and its off-site detections spread down to the edge midpoint. Subtracting real heavy columns leaves structured left overs that the synthetic residuals did not have. It is also weaker at faint Cu (F1 0.828 at Cu/heavy 0.15).
* **V2a is not flawless:** Cu2 : Cu1 ranges from 1.87 (1122) to 2.34 (1709), 1333 has 4% more Cu than heavy, and up to 945 Cu per image (1530 out of 57,000) cannot be classified as Cu1 or Cu2.
* **Trained on synthetic data only:** anything the renderer does not model (defects, amorphous regions, scan distortion) is unseen.
* Real images have no ground truth right now; they are judged by the Cu:heavy and Cu2:Cu1 ratio counts, the Cu site positions, and the overlays.
* Agreement between V2b and my AtomAI Cu labels is partly built in since both come from a residual; V2a is the independent comparison.

## What this is not testing / What the results don't show
* **Column-by-column accuracy on real images.** Every real Cu site is occupied, so a model that marks every Cu site and a true Cu detector give the same Cu:heavy and Cu2:Cu1 ratio counts. The evidence that V2a uses brightness comes from the synthetic negative control; the real counts are consistent with it, but they are not independent proof.
* **Realistic "missing" Cu:** The control's empty sites are idealized (meaning nothing drawn there).  Partly occupied, disordered, or unusually dim real columns are only tested indirectly through the brightness sweep.
* **Generalization beyond the generator:** The synthetic test sets come from the same generator as training (different seeds), so those scores are overly optimistic. Defects, interfaces, amorphous regions, other zone axes, other materials, and pixel sizes outside 0.119–0.477 Å are not tested.
* **The faintest individual columns:** Brightness was measured only as cell averages, so the spread of individual real Cu columns is unknown; V2a's F1 falls to 0.941 at Cu/heavy 0.15.
* **Y vs Ba, and oxygen:** Y and Ba are one class; oxygen is labelled background and never detected.

In [ ]:
  # Results at a glance (re-run this cell after the full run)
from pathlib import Path
from IPython.display import Markdown, display
_results = Path("shapenet_ybco_outputs") / "RESULTS.md"
if _results.exists():
    display(Markdown(_results.read_text()))
else:
    print("No results yet: run the notebook to the end, then re-run this cell.")

### Setup
Every constant states where it comes from: **[Phase 2]** = copied from *YBCO_Phase2_AtomAI_UNet_v2_5* cell 5,
**[measured]** = measured in Section A or the Calibration section of this notebook, **[crystal]** = YBa₂Cu₃O₇ structure,
**[ShapeOp]** = the shared *Shape Operator Three Experiments* notebook, **[choice]** = a design choice, not tuned.

In [ ]:
# Imports

import json, math, time, copy, re
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import scipy.ndimage as ndi
from scipy.ndimage import gaussian_filter, gaussian_filter1d, zoom, maximum_filter
from scipy.optimize import least_squares, linear_sum_assignment
from scipy.spatial import cKDTree
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import hashlib
import platform
import shutil
import sys
import scipy
import matplotlib

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True   # fastest convolution algorithms; GPU training is not bit-reproducible anyway (grid_sample backward uses atomic adds)
if DEVICE == "cpu":
    print("WARNING: no GPU. Training here takes about 17 s per batch of 8 on a 2-core CPU (about 1.8 h per epoch at the "
          "default sizes). In Colab: Runtime -> Change runtime type -> GPU, then run all cells again.")
OUTPUT_DIR = Path("shapenet_ybco_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)

# --- images -------------------------------------------------------------------------------
DROP_IDS = ["1717", "1338", "1520", "1734"]     # Decidedd not to use images due to unuseable regions in them

# --- scale and labels ------------------------------------------------------------
CANON_PERIOD = 16.0          # px per in-plane lattice period after resampling
ATOM_RADIUS_REL = 0.22       # label disk radius = 3.52 px
BACKGROUND, HEAVY, CU = 0, 1, 2
CU_FAINT_CODE = 102          # unused right now (no faint sub-population is defined for the synthetic data)
IGNORE_CODE = 255
VACANT_CODE = 50             # temporary label for empty Cu sites; becomes background after augmentation
CROP = 256                   # crop size: 256 x 256

# --- augmentation ------------------------------
AUG_PER_CROP = 2
AUG_SCALE_JITTER = 0.15
AUG_ROTATION_DEG = 10.0
AUG_GAMMA = (0.85, 1.18)
AUG_CONTRAST = (0.85, 1.18)
AUG_NOISE_SIGMA = (0.0, 0.03)
AUG_BLUR_SIGMA_REL = (0.0, 0.15)

# --- evaluation -----------------------------------------------------------------
MATCH_TOL = 0.25 * CANON_PERIOD          # MATCH_TOLERANCE_REL = 0.25 -> 4 px
TOLERANCE_SWEEP_REL = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
MIN_PEAK_DIST = 0.25 * CANON_PERIOD      # 0.25 leaves approx. 0.00% of true Cu unresolvable
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]   # chosen on validation, then frozen
BATCH_SIZE = 8

# --- training [ShapeOp] / [choice] --------------------------------------------------------
LEARNING_RATE = 3e-3        # [ShapeOp] Adam learning rate
EPOCHS = 15                 # [choice] halved from 30 for sake of run time
N_TRAIN, N_VAL, N_TEST = 3000, 300, 300   # [choice]; trained on 3,336 arrays
SWEEP_RATIOS = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75]   # Cu/heavy values for the detection-limit test
N_PER_RATIO = 40            # [choice]
N_CTRL = 40                 # [choice] crops per negative-control set
CTRL_CU_RATIO = 0.52        # [measured] median Cu/heavy in the real average cells (0.516)
SEEDS = dict(train=110, val=220, test=330, sweep=440, res_train=550, res_val=660, res_test=770, ctrl_absent=880, ctrl_half=990)   # [ShapeOp] style
DIST_SCALE = 0.5 * CANON_PERIOD          # [choice] distance-to-center, target saturates at half a period
print("device:", DEVICE, "| torch", torch.__version__)

### A. Lattice characterization of the YBCO images
How the periods, zone axis, intensity ratios, and image quality were obtained:

1. **Pixel size:** Read from every `.emd` file's metadata.
2. **Periods (FFT):** The 2D Fourier transform of a periodic image has spots at the lattice frequencies; a spot at
   spatial frequency **k** means a spacing of **1/k**. Spot centers are refined by a weighted centroid.
3. **Average unit cell (folding):** With the two FFT lattice vectors, every pixel gets a fractional position (u, v)
   in the cell, and averaging all pixels with the same (u, v) folds thousands of cells into one low-noise average cell.
4. **Columns in the cell:** 6 local maxima are expected (3 heavy + 3 Cu is the ratio). They split into two sub-lattices half a
   period apart; the brighter one is the heavy column. Species are assigned by **geometry and not brightness**.
5. **Ratios:** Peak heights above the cell minimum. These are drift-blurred averages, which pull ratios toward 1.

In [ ]:
def pixel_size_A(file):
    """Pixel size in Angstrom from the Velox .emd metadata."""
    with h5py.File(file, "r") as h:
        grp = h["Data/Image"]
        for uid in grp:
            md = json.loads(bytes(np.array(grp[uid]["Metadata"][:, 0])).decode("utf-8", "ignore").rstrip("\x00"))
            if "HAADF" in str(md.get("BinaryResult", {}).get("Detector", "")).upper():
                return float(md["BinaryResult"]["PixelSize"]["width"]) * 1e10
        uid = next(iter(grp))
        md = json.loads(bytes(np.array(grp[uid]["Metadata"][:, 0])).decode("utf-8", "ignore").rstrip("\x00"))
        return float(md["BinaryResult"]["PixelSize"]["width"]) * 1e10


def _refine_peak(P, k, kx0, ky0, r):
    KX, KY = np.meshgrid(k, k); m = np.hypot(KX - kx0, KY - ky0) < r
    iy, ix = np.unravel_index(np.argmax(np.where(m, P, -1)), P.shape)
    sl = (slice(iy - 2, iy + 3), slice(ix - 2, ix + 3)); w = P[sl] - P[sl].min()
    return np.array([(KX[sl] * w).sum() / w.sum(), (KY[sl] * w).sum() / w.sum()])


def lattice_vectors(img, a_guess_px, c_guess_px):
    """Reciprocal vectors (cycles/px): g1 for the in-plane period, g2 for c (refined on its 3rd harmonic)."""
    n = min(img.shape); a = img[:n, :n].astype(np.float64); a = (a - a.mean()) / a.std()
    P = np.abs(np.fft.fftshift(np.fft.fft2(a * np.outer(np.hanning(n), np.hanning(n))))) ** 2
    k = (np.arange(n) - n // 2) / n
    g1 = _refine_peak(P, k, 1 / a_guess_px, 0, 0.25 / a_guess_px); g1 = g1 if g1[0] > 0 else -g1
    g2 = _refine_peak(P, k, 0, 3 / c_guess_px, 0.6 / c_guess_px) / 3
    return g1, g2


NU, NV = 24, 72
def fold_cell(crop, a_guess_px, c_guess_px):
    g1, g2 = lattice_vectors(crop, a_guess_px, c_guess_px)
    yy, xx = np.mgrid[:crop.shape[0], :crop.shape[1]].astype(np.float64)
    u = (xx * g1[0] + yy * g1[1]) % 1; v = (xx * g2[0] + yy * g2[1]) % 1
    H, _, _ = np.histogram2d(v.ravel(), u.ravel(), bins=[NV, NU], range=[[0, 1], [0, 1]], weights=crop.ravel())
    C, _, _ = np.histogram2d(v.ravel(), u.ravel(), bins=[NV, NU], range=[[0, 1], [0, 1]])
    return H / np.maximum(C, 1), 1 / np.hypot(*g1), 1 / np.hypot(*g2)


def _cdist(a, b):
    d = abs(a - b) % 1; return min(d, 1 - d)


def measure_cell(cell):
    """Locate the 6 columns, assign species by geometry, return ratios (or None if the fold is not clean)."""
    t = gaussian_filter(np.tile(cell, (3, 3)), 1.0); sub = t[NV:2 * NV, NU:2 * NU]
    mx = maximum_filter(t, size=(9, 5))[NV:2 * NV, NU:2 * NU]
    pk = []
    for y, x in zip(*np.nonzero(sub == mx)):
        Y, X = y + NV, x + NU
        dy = 0.5 * (t[Y - 1, X] - t[Y + 1, X]) / (t[Y - 1, X] - 2 * t[Y, X] + t[Y + 1, X] + 1e-12)
        dx = 0.5 * (t[Y, X - 1] - t[Y, X + 1]) / (t[Y, X - 1] - 2 * t[Y, X] + t[Y, X + 1] + 1e-12)
        pk.append(((x + dx + 0.5) / NU % 1, (y + dy + 0.5) / NV % 1, t[Y, X]))
    if len(pk) != 6:
        return None, len(pk)
    us, vs, I = (np.array(z) for z in zip(*pk))
    ref = us[np.argmax(I)]; g = np.array([_cdist(u, ref) < 0.25 for u in us])
    if g.sum() != 3:
        return None, len(pk)
    bg = sub.min(); Ih, Ic, vh, vc, uh, uc = I[g] - bg, I[~g] - bg, vs[g], vs[~g], us[g], us[~g]
    o = np.argsort(vh); vh, Ih, uh = vh[o], Ih[o], uh[o]
    gaps = [(vh[(i + 1) % 3] - vh[i]) % 1 for i in range(3)]
    big = int(np.argmax(gaps)); y_idx = (big + 2) % 3; ba = [i for i in range(3) if i != y_idx]
    mid = (vh[big] + gaps[big] / 2) % 1; cu1 = int(np.argmin([_cdist(v, mid) for v in vc]))
    return dict(gaps=sorted(gaps), Y_Ba=float(Ih[y_idx] / Ih[ba].mean()), Cu_Ba=float(Ic.mean() / Ih[ba].mean()),
                Cu_heavy=float(Ic.mean() / Ih.mean()), Cu1_Cu2=float(Ic[cu1] / np.delete(Ic, cu1).mean()),
                heavy_uv=np.c_[uh, vh], cu_uv=np.c_[uc, vc]), 6

In [ ]:
# Characterize ALL 18 images (this is the evidence used to drop four of them).
REGION_A = 80.0                                         # [choice] 8 nm square regions
REGION_CENTRES = [(512, 512), (1024, 1024), (1536, 1536), (512, 1536)]
rows = []

for file, raw in zip(ybco_files, ybco_images):
    px = pixel_size_A(file); img = np.asarray(raw, np.float64)
    n = int(REGION_A / px); n -= n % 2
    for ci, (cy, cx) in enumerate(REGION_CENTRES):
        cy = min(max(cy, n // 2), img.shape[0] - n // 2); cx = min(max(cx, n // 2), img.shape[1] - n // 2)
        cell, a_px, c_px = fold_cell(img[cy - n // 2:cy + n // 2, cx - n // 2:cx + n // 2], 3.99 / px, 12.2 / px)
        m, npk = measure_cell(cell)
        r = dict(image=file.stem, id=file.stem[:4], region=ci, px_A=px, a_A=a_px * px, c_A=c_px * px, peaks=npk, clean=m is not None)
        if m is not None:
            r.update({k: m[k] for k in ["Y_Ba", "Cu_Ba", "Cu_heavy", "Cu1_Cu2"]}); r["gaps"] = m["gaps"]
            # a fold is only accepted if Y/Ba has heavy assignment consistency
            r["clean"] = bool(0.5 < m["Y_Ba"] < 1.0 and max(m["gaps"]) < 0.43)
        rows.append(r)

fold_df = pd.DataFrame(rows)
quality = fold_df.groupby("id")["clean"].agg(["sum", "count"]).rename(columns={"sum": "clean_regions", "count": "regions"})
print(quality.T.to_string())
print("\nin-plane period %.3f +- %.3f A;  c period %.2f-%.2f A" % (fold_df.a_A.mean(), fold_df.a_A.std(), fold_df.c_A.min(), fold_df.c_A.max()))

In [ ]:
# Measured quantities over clean regions of the 14 kept images
kept_clean = fold_df[fold_df.clean & ~fold_df.id.isin(DROP_IDS)]
print(f"{len(kept_clean)} clean regions in the 14 kept images")

for k in ["Y_Ba", "Cu_Ba", "Cu_heavy", "Cu1_Cu2"]:
    v = kept_clean[k]; print(f"{k:9s} median {v.median():.3f}  IQR {v.quantile(.25):.3f}-{v.quantile(.75):.3f}  range {v.min():.3f}-{v.max():.3f}")

g = np.array(kept_clean.gaps.tolist())
print("heavy gaps along c (fraction of c), median:", np.round(np.median(g, 0), 3), " ideal YBCO-123: [0.316 0.316 0.368]")
for name, ratio, z in [("Y/Ba", kept_clean.Y_Ba.median(), 39 / 56), ("Cu/Ba", kept_clean.Cu_Ba.median(), 29 / 56)]:
    print(f"exponent n implied by {name} = Z^n: {np.log(ratio) / np.log(z):.2f}   (Z^1.7 would give {z ** 1.7:.2f})")

**Cu site check against crystallography.** In ideal YBa₂Cu₃O₇ viewed along [100] (a = 3.82, b = 3.885, c = 11.68 Å;
Ba z = 0.184, Y z = 0.5, Cu1 z = 0, Cu2 z = 0.355; researched), each Cu sits inside a quadrilateral of heavy columns. The cell
below computes the nearest Cu to heavy atomic column distance divided by the heavy to heavy atomic column nearest-neighbour distance to obtain the ideal
structure and for the measured average cells.

In [ ]:
def _site_ratios(heavy_uv, cu_uv, a, c):
    H = [(u + i, v + j) for (u, v) in heavy_uv for i in (-1, 0, 1) for j in (-1, 0, 1)]
    d = lambda p, q: np.hypot((p[0] - q[0]) * a, (p[1] - q[1]) * c)
    hh = min(d(p, q) for p in heavy_uv for q in H if d(p, q) > 1e-6)
    return hh, sorted(min(d(p, q) for q in H) / hh for p in cu_uv)

hh, ideal = _site_ratios([(0.5, .184), (0.5, .5), (0.5, .816)], [(0, 0), (0, .355), (0, .645)], 3.885, 11.68)
print("ideal  [100]: heavy NN %.2f A; Cu-heavy / heavy-NN = %s" % (hh, np.round(ideal, 3)))
meas = []

for _, r in kept_clean.iterrows():
    file = [f for f in ybco_files if f.stem == r.image][0]; raw = ybco_images[ybco_files.index(file)]
    px = r.px_A; n = int(REGION_A / px); n -= n % 2; cy, cx = REGION_CENTRES[r.region]
    cy = min(max(cy, n // 2), raw.shape[0] - n // 2); cx = min(max(cx, n // 2), raw.shape[1] - n // 2)
    cell, a_px, c_px = fold_cell(np.asarray(raw, np.float64)[cy - n // 2:cy + n // 2, cx - n // 2:cx + n // 2], 3.99 / px, 12.2 / px)
    m, _ = measure_cell(cell)
    meas.append(_site_ratios(m["heavy_uv"], m["cu_uv"], a_px * px, c_px * px)[1])
meas = np.array(meas)

print("measured     : Cu-heavy / heavy-NN median = %s over %d regions" % (np.round(np.median(meas, 0), 3), len(meas)))

### B. Keep 14 images and resample them to the canonical scale
Each image is resampled so its in-plane period is `CANON_PERIOD` = 16 px, with the Phase 2 image path
(`normalize_image`, Gaussian anti-alias before shrinking, `zoom(order=1)`). The factor is stored per image and passed
explicitly — never through a global — so nothing downstream can silently use a stale scale.

**Coordinate mapping.** `scipy.ndimage.zoom` (default `grid_mode=False`) aligns the first and last pixels, so a
native coordinate *x* lands at *x*·(n_out−1)/(n_in−1), not at *x*·factor. The difference reaches about one canonical
pixel at the far edge (tested below). Phase 2 v2.5 scales label coordinates by `factor`, so its labels carry this
offset; it is noted here for the later comparison and avoided in this notebook.

In [ ]:
def normalize_image(img, p_low=0.5, p_high=99.5):
    """Verbatim from YBCO_Phase2_AtomAI_UNet_v2_5, cell 9."""
    img = np.asarray(img, dtype=np.float32)
    lo, hi = np.percentile(img, [p_low, p_high])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0.0, 1.0).astype(np.float32)


def to_canonical(raw, factor):
    """The Phase 2 resample_scan image path."""
    img = normalize_image(raw)
    if factor < 1.0:
        img = gaussian_filter(img, sigma=0.5 * (1.0 / factor - 1.0))
    return zoom(img, factor, order=1).astype(np.float32)


def canonical_to_native(pts, native_shape, canon_shape):
    out = np.array(pts, dtype=float).reshape(-1, 2).copy()
    out[:, 0] *= (native_shape[1] - 1) / (canon_shape[1] - 1)
    out[:, 1] *= (native_shape[0] - 1) / (canon_shape[0] - 1)
    return out


# test of the mapping with an impulse
for f in [1.349, 0.480]:
    a = np.zeros(2048, np.float32); a[2000] = 1; z = zoom(a, f, order=1)
    landed = (np.arange(len(z)) * z).sum() / z.sum()
    print(f"factor {f}: impulse at native 2000 lands at {landed:.3f}; x*factor = {2000 * f:.3f}; x*(n_out-1)/(n_in-1) = {2000 * (len(z) - 1) / 2047:.3f}")

keep = [(f, im) for f, im in zip(ybco_files, ybco_images) if f.stem[:4] not in DROP_IDS]
assert len(keep) == 14, len(keep)
scan_rows, canon = [], {}
for f, raw in keep:
    raw = np.asarray(raw); px = pixel_size_A(f)
    g1, g2 = lattice_vectors(raw.astype(np.float64), 3.99 / px, 12.2 / px)
    a_px, c_px = 1 / np.hypot(*g1), 1 / np.hypot(*g2)
    factor = CANON_PERIOD / a_px
    canon[f.stem] = to_canonical(raw, factor)
    scan_rows.append(dict(image=f.stem, px_A=px, a_px=a_px, a_A=a_px * px, c_A=c_px * px, c_over_a=c_px / a_px,
                          factor=factor, native_shape=raw.shape, canon_shape=canon[f.stem].shape))
scan_df = pd.DataFrame(scan_rows).set_index("image")
# the canonical period must now be 16 px on every image
for name, im in canon.items():
    g1c, _ = lattice_vectors(im[:1024, :1024].astype(np.float64), CANON_PERIOD, scan_df.loc[name, "c_over_a"] * CANON_PERIOD)
    assert abs(1 / np.hypot(*g1c) - CANON_PERIOD) < 0.3, (name, 1 / np.hypot(*g1c))
print(scan_df[["px_A", "a_px", "a_A", "c_A", "c_over_a", "factor", "canon_shape"]].round(3).to_string())

### C. Synthetic YBCO generator (crystal structure → Phase 2 alteration)
`augment_pair` and `stamp_disks` below are **copied verbatim** from Phase 2 v2.5 (cells 13 and 9). The renderer adds
only what the augmenter does not model; every range is listed with its source.

| Range | Value | Source |
|---|---|---|
| c / a | 2.92–3.22 | [measured] 2.949–3.192 on the 14 images (Section B) |
| Ba height above Cu1 | 0.180–0.195 c | [crystal] 0.184; [measured] 0.19 (half the measured 0.381 c Ba–Ba gap) |
| Cu2 height | 0.345–0.365 c | [crystal] 0.355 |
| lattice rotation / shear | ±3° / ±2° | [measured] a-axis rotation −2.8° to +1.3° in the FFT fits |
| Y / Ba | 0.75–1.00 | [measured] median 0.87, IQR 0.85–0.89 |
| Cu / heavy | **0.15–0.75** | the sweep; [measured] median 0.52 in cell averages, which hide individual faint columns |
| Cu1 / Cu2 | 0.55–0.95 | [measured] median 0.76, range 0.55–0.96 |
| O / heavy | 0–0.10, **labelled background** | the 53 clean average cells show only the 6 cation maxima |
| column σ (render) | 2.4–3.0 px | calibrated in the Calibration section so rendered + augmenter blur matches the measured σ |
| noise σ (pre-normalisation) | 0.06–0.14 | calibrated in the Calibration section against the measured noise metric |
| Poisson dose at a Ba peak | 30–400 counts | [choice] spans low to moderate dose |
| per-column intensity / position scatter | 8% / 0.35 px | [choice] |
| thickness field / flat background | 0–0.20 / 0.05–0.30 | [choice] |
| scan-line jitter | 0–0.4 px | [choice] |
| **Cu vacancies** | 0–50% of Cu columns removed per crop; 10% of crops with no Cu; empty sites labelled background | [choice] training device, not physics: without it every Cu site is filled and a model can place Cu from the heavy lattice alone |

In [ ]:
def augment_pair(image, label, rng):
    """Verbatim from YBCO_Phase2_AtomAI_UNet_v2_5, cell 13."""
    img = np.asarray(image, dtype=np.float32)
    lab = np.asarray(label, dtype=np.uint8)

    # scale jitter - absorbs error in the period measurement
    if AUG_SCALE_JITTER > 0:
        s = float(rng.uniform(1.0 - AUG_SCALE_JITTER, 1.0 + AUG_SCALE_JITTER))
        if abs(s - 1.0) > 1e-3:
            if s < 1.0:
                img = gaussian_filter(img, sigma=0.5 * (1.0 / s - 1.0))
            img = zoom(img, s, order=1)
            lab = zoom(lab, s, order=0)
            H, W = img.shape
            if H >= CROP and W >= CROP:
                y0 = int(rng.integers(0, H - CROP + 1)); x0 = int(rng.integers(0, W - CROP + 1))
                img = img[y0:y0+CROP, x0:x0+CROP]; lab = lab[y0:y0+CROP, x0:x0+CROP]
            else:
                pi = np.zeros((CROP, CROP), np.float32)
                pl = np.full((CROP, CROP), IGNORE_CODE, np.uint8)
                pi[:H, :W] = img; pl[:H, :W] = lab
                img, lab = pi, pl

    if rng.random() < 0.5:
        img, lab = np.fliplr(img).copy(), np.fliplr(lab).copy()
    if rng.random() < 0.5:
        img, lab = np.flipud(img).copy(), np.flipud(lab).copy()
    k = int(rng.integers(0, 4))
    if k:
        img, lab = np.rot90(img, k).copy(), np.rot90(lab, k).copy()

    if AUG_ROTATION_DEG > 0:
        ang = float(rng.uniform(-AUG_ROTATION_DEG, AUG_ROTATION_DEG))
        if abs(ang) > 0.2:
            img = ndi.rotate(img, ang, reshape=False, order=1, mode="reflect")
            lab = ndi.rotate(lab, ang, reshape=False, order=0, mode="constant",
                             cval=IGNORE_CODE)

    b = float(rng.uniform(*AUG_BLUR_SIGMA_REL)) * CANON_PERIOD
    if b > 0.05:
        img = gaussian_filter(img, sigma=b)
    img = np.clip(img, 0, 1) ** float(rng.uniform(*AUG_GAMMA))
    c = float(rng.uniform(*AUG_CONTRAST))
    img = np.clip((img - img.mean()) * c + img.mean(), 0, 1)
    s = float(rng.uniform(*AUG_NOISE_SIGMA))
    if s > 0:
        img = np.clip(img + rng.normal(0, s, img.shape), 0, 1)

    return img.astype(np.float32), lab.astype(np.uint8)


def stamp_disks(shape, coords, radius, value, out=None):
    """Verbatim from YBCO_Phase2_AtomAI_UNet_v2_5, cell 9."""
    mask = np.zeros(shape, dtype=np.uint8) if out is None else out
    if len(coords) == 0 or radius <= 0:
        return mask
    h, w = shape
    r2 = float(radius) ** 2
    for x, y in np.asarray(coords, dtype=float).reshape(-1, 2):
        x0, x1 = max(0, int(np.floor(x - radius))), min(w, int(np.ceil(x + radius)) + 1)
        y0, y1 = max(0, int(np.floor(y - radius))), min(h, int(np.ceil(y + radius)) + 1)
        if x1 <= x0 or y1 <= y0:
            continue
        yy, xx = np.ogrid[y0:y1, x0:x1]
        mask[y0:y1, x0:x1][((xx - x) ** 2 + (yy - y) ** 2) <= r2] = value
    return mask

In [ ]:
GEN = dict(c_over_a=(2.92, 3.22), ba_z=(0.180, 0.195), cu2_z=(0.345, 0.365), lattice_rot_deg=(-3.0, 3.0),
           shear_deg=(-2.0, 2.0), y_over_ba=(0.75, 1.00), cu_over_heavy=(0.15, 0.75), cu1_over_cu2=(0.55, 0.95),
           o_over_heavy=(0.0, 0.10), column_jitter=0.08, position_jitter_px=0.35, sigma_px=(2.4, 3.0),
           background=(0.05, 0.30), thickness_amp=(0.0, 0.20), noise_sigma=(0.06, 0.14), dose=(30.0, 400.0),
           line_jitter_px=(0.0, 0.4), cu_vacancy=(0.0, 0.5), p_no_cu=0.10)


def cell_sites(view, zb, zc2):
    """Projected (u, v) of resolvable columns. view '100': along a; '010': along b (YBCO twins, both occur)."""
    heavy = [(0.5, zb, "Ba"), (0.5, 0.5, "Y"), (0.5, 1 - zb, "Ba")]
    cu = [(0.0, 0.0, "Cu1"), (0.0, zc2, "Cu2"), (0.0, 1 - zc2, "Cu2")]
    if view == "100":   # O1 (chain), O3 (plane), O4 (apical); O2 projects ~0.3 A from Cu2 and merges with it
        oxy = [(0.5, 0.0), (0.5, 0.377), (0.5, 0.623), (0.0, 0.159), (0.0, 0.841)]
    else:               # O2 (plane), O4 (apical); O1 projects onto Cu1, O3 ~0.3 A from Cu2
        oxy = [(0.5, 0.378), (0.5, 0.622), (0.0, 0.159), (0.0, 0.841)]
    return heavy, cu, oxy


def render_clean(rng, size=CROP, cu_ratio=None, vacancy=None):
    g = GEN; a = CANON_PERIOD; c = a * rng.uniform(*g["c_over_a"])
    zb = rng.uniform(*g["ba_z"]); zc2 = rng.uniform(*g["cu2_z"]); view = "100" if rng.random() < 0.5 else "010"
    rot = np.radians(rng.uniform(*g["lattice_rot_deg"])); shear = np.radians(rng.uniform(*g["shear_deg"]))
    e1 = a * np.array([np.cos(rot), np.sin(rot)]); e2 = c * np.array([-np.sin(rot + shear), np.cos(rot + shear)])
    origin = rng.uniform(0, 1, 2) * np.array([a, c])
    r_y = rng.uniform(*g["y_over_ba"]); r_cu = rng.uniform(*g["cu_over_heavy"]) if cu_ratio is None else float(cu_ratio)
    r_c12 = rng.uniform(*g["cu1_over_cu2"]); r_o = rng.uniform(*g["o_over_heavy"]); sig = rng.uniform(*g["sigma_px"])
    mean_heavy = (2.0 + r_y) / 3.0
    amp = {"Ba": 1.0, "Y": r_y, "Cu2": r_cu * mean_heavy, "Cu1": r_cu * mean_heavy * r_c12}
    heavy_s, cu_s, o_s = cell_sites(view, zb, zc2)
    pad = 3 * a; n1 = int(np.ceil((size + 2 * pad) / a)) + 2; n2 = int(np.ceil((size + 2 * pad) / c)) + 2
    pts = []
    for i in range(-n1, n1 + 1):
        for j in range(-n2, n2 + 1):
            for (u, v, sp) in heavy_s:
                p = origin + (i + u) * e1 + (j + v) * e2; pts.append((p[0], p[1], amp[sp], 1))
            for (u, v, sp) in cu_s:
                p = origin + (i + u) * e1 + (j + v) * e2; pts.append((p[0], p[1], amp[sp], 2))
            for (u, v) in o_s:
                p = origin + (i + u) * e1 + (j + v) * e2; pts.append((p[0], p[1], r_o * mean_heavy, 0))
    P = np.array(pts, dtype=float)
    P = P[(P[:, 0] > -pad) & (P[:, 0] < size + pad) & (P[:, 1] > -pad) & (P[:, 1] < size + pad)]
    # Cu vacancies - a training device, not a physical claim: a random fraction of Cu columns is removed and its sites
    # are labelled background, so a Cu label is only correct where Cu brightness is actually present
    f_vac = vacancy if vacancy is not None else (1.0 if rng.random() < g["p_no_cu"] else rng.uniform(*g["cu_vacancy"]))
    cu_rows = np.nonzero(P[:, 3] == 2)[0]
    P[cu_rows[rng.random(len(cu_rows)) < f_vac], 3] = 3          # kind 3 = vacant Cu site
    P[:, :2] += rng.normal(0, g["position_jitter_px"], (len(P), 2))
    P[:, 2] *= np.clip(rng.normal(1, g["column_jitter"], len(P)), 0.5, 1.5)
    jit = rng.uniform(*g["line_jitter_px"])
    rs = gaussian_filter1d(rng.normal(0, 1, size + 2 * int(pad)), 2.0); rs = jit * rs / (rs.std() + 1e-9)
    P[:, 0] += rs[np.clip(np.round(P[:, 1] + pad).astype(int), 0, len(rs) - 1)]
    img = np.zeros((size, size), np.float64); w = int(np.ceil(4 * sig))
    for x, y, A, kind in P:
        if kind == 3:
            continue                                        # vacant Cu site: nothing rendered
        x0, x1 = max(0, int(x) - w), min(size, int(x) + w + 2); y0, y1 = max(0, int(y) - w), min(size, int(y) + w + 2)
        if x1 <= x0 or y1 <= y0:
            continue
        gx = np.exp(-((np.arange(x0, x1) - x) ** 2) / (2 * sig * sig)); gy = np.exp(-((np.arange(y0, y1) - y) ** 2) / (2 * sig * sig))
        img[y0:y1, x0:x1] += A * np.outer(gy, gx)
    th = gaussian_filter(rng.normal(0, 1, (size, size)), size / 4); th = rng.uniform(*g["thickness_amp"]) * th / (th.std() + 1e-9)
    img = img * (1 + th) + rng.uniform(*g["background"]) * (1 + th)
    dose = rng.uniform(*g["dose"]); img = rng.poisson(np.clip(img, 0, None) * dose) / dose
    img = img + rng.normal(0, rng.uniform(*g["noise_sigma"]), img.shape)
    radius = ATOM_RADIUS_REL * CANON_PERIOD
    inside = (P[:, 0] > -radius) & (P[:, 0] < size + radius) & (P[:, 1] > -radius) & (P[:, 1] < size + radius)
    label = np.zeros((size, size), np.uint8)
    stamp_disks((size, size), P[inside & (P[:, 3] == 1), :2], radius, HEAVY, out=label)
    stamp_disks((size, size), P[inside & (P[:, 3] == 2), :2], radius, CU, out=label)
    stamp_disks((size, size), P[inside & (P[:, 3] == 3), :2], radius, VACANT_CODE, out=label)
    return img.astype(np.float32), label, dict(view=view, vacancy=f_vac, cu_ratio=r_cu, y_over_ba=r_y, cu1_over_cu2=r_c12,
                                               o_ratio=r_o, sigma_px=sig, dose=dose)


def centres_from_label(label, code):
    """Column centres from the (augmented) label disks; disks clipped by the border or the ignore region are 'partial'."""
    m = label == code; lab, n = ndi.label(m)
    if n == 0:
        return np.zeros((0, 2)), np.zeros((0, 2))
    idx = np.arange(1, n + 1); area = ndi.sum(m, lab, idx)
    cy, cx = np.array(ndi.center_of_mass(m, lab, idx)).T
    touch_ign = ndi.maximum(ndi.binary_dilation(label == IGNORE_CODE), lab, idx) > 0
    border = np.zeros_like(m); border[[0, -1], :] = True; border[:, [0, -1]] = True
    touch_b = ndi.maximum(border, lab, idx) > 0
    full = np.pi * (ATOM_RADIUS_REL * CANON_PERIOD) ** 2
    partial = touch_ign | touch_b | (area < 0.6 * full * (1 - AUG_SCALE_JITTER) ** 2)
    return np.c_[cx[~partial], cy[~partial]], np.c_[cx[partial], cy[partial]]


def make_sample(rng, augment=True, cu_ratio=None, vacancy=None):
    img, lab, meta = render_clean(rng, CROP, cu_ratio=cu_ratio, vacancy=vacancy)
    img = normalize_image(img)
    if augment:
        img, lab = augment_pair(img, lab, rng)
    heavy, heavy_p = centres_from_label(lab, HEAVY); cu, cu_p = centres_from_label(lab, CU)
    vac, vac_p = centres_from_label(lab, VACANT_CODE)                 # empty Cu sites, tracked through the augmenter
    lab = lab.copy(); lab[lab == VACANT_CODE] = BACKGROUND
    meta["augmented"] = augment
    # distance from every pixel to the nearest centre (full and partial disks), / DIST_SCALE, clipped at 1;
    # computed once here (float16) instead of on every epoch
    H, W = img.shape; yy, xx = np.mgrid[:H, :W]; q = np.c_[xx.ravel(), yy.ravel()]; dist = {}
    for key, pts in [("heavy", np.r_[heavy, heavy_p]), ("cu", np.r_[cu, cu_p])]:
        dist[key] = (np.ones((H, W), np.float16) if len(pts) == 0 else
                     np.minimum(cKDTree(pts).query(q)[0].reshape(H, W) / DIST_SCALE, 1.0).astype(np.float16))
    return dict(image=img, label=lab, heavy=heavy, cu=cu, heavy_partial=heavy_p, cu_partial=cu_p, cu_vacant=vac, cu_vacant_partial=vac_p, dist=dist, meta=meta)


def make_dataset(n, seed, cu_ratio=None, vacancy=None):
    """1 raw : AUG_PER_CROP augmented, mirroring the Phase 2 training mix."""
    rng = np.random.default_rng(seed)
    return [make_sample(rng, augment=(i % (1 + AUG_PER_CROP)) != 0, cu_ratio=cu_ratio, vacancy=vacancy) for i in range(n)]

### Calibration: same measurements on real and synthetic crops
Three metrics are applied identically to 256 × 256 canonical crops: **column σ** (free 2D Gaussian fits on the brightest
maxima), **noise** (standard deviation of the crop minus its own periodic average, slow drift removed) and
**contrast** (99th − 5th percentile). The render σ and noise ranges in `GEN` were set so the synthetic distribution
brackets the real one. Crops padded by the augmenter's jitter are skipped.

In [ ]:
def _fft_vectors_any(reg):
    n = reg.shape[0]; a = (reg - reg.mean()) / reg.std()
    P = np.abs(np.fft.fftshift(np.fft.fft2(a * np.outer(np.hanning(n), np.hanning(n))))) ** 2
    k = (np.arange(n) - n // 2) / n; KX, KY = np.meshgrid(k, k); K = np.hypot(KX, KY)
    def best(mask):
        iy, ix = np.unravel_index(np.argmax(np.where(mask, P, -1)), P.shape)
        sl = (slice(iy - 2, iy + 3), slice(ix - 2, ix + 3)); w = P[sl] - P[sl].min()
        return np.array([(KX[sl] * w).sum() / w.sum(), (KY[sl] * w).sum() / w.sum()])
    g1 = best((K > 1 / 19) & (K < 1 / 13.5) & (KY >= 0))
    perp = np.array([-g1[1], g1[0]]) / np.hypot(*g1); proj = KX * perp[0] + KY * perp[1]
    g3 = best((K > 3 / 54) & (K < 3 / 44) & (np.abs(proj / np.maximum(K, 1e-9)) > 0.97) & (proj > 0))
    return g1, g3 / 3


def noise_metric(reg, nu=16, nv=48):
    g1, g2 = _fft_vectors_any(reg)
    yy, xx = np.mgrid[:reg.shape[0], :reg.shape[1]].astype(float)
    u = (xx * g1[0] + yy * g1[1]) % 1; v = (xx * g2[0] + yy * g2[1]) % 1
    idx = np.minimum((v * nv).astype(int), nv - 1) * nu + np.minimum((u * nu).astype(int), nu - 1)
    s = np.bincount(idx.ravel(), reg.ravel(), nu * nv); c = np.bincount(idx.ravel(), None, nu * nv)
    res = reg - (s / np.maximum(c, 1))[idx]
    return float(np.std(res - gaussian_filter(res, 6)))


_YG, _XG = np.mgrid[-5:6, -5:6].astype(float)
def gauss_fit_sigma(patch, x0=0.0, y0=0.0):
    r = least_squares(lambda q: (q[0] * np.exp(-((_XG - q[1]) ** 2 + (_YG - q[2]) ** 2) / (2 * q[3] ** 2)) + q[4] - patch).ravel(),
                      [patch.max() - patch.min(), x0, y0, 2.5, patch.min()], bounds=([0, -2, -2, .5, -1], [3, 2, 2, 6, 2]))
    return r.x[3]


def sigma_metric(reg):
    sm = gaussian_filter(reg, 1.0)
    ys, xs = np.nonzero((sm == maximum_filter(sm, size=9)) & (sm > np.percentile(sm, 97)))
    out = [gauss_fit_sigma(reg[y - 5:y + 6, x - 5:x + 6]) for y, x in zip(ys, xs)
           if 6 <= y <= reg.shape[0] - 7 and 6 <= x <= reg.shape[1] - 7]
    return float(np.median(out)) if out else np.nan


calib = []
for name, im in canon.items():
    H, W = im.shape
    for (cy, cx) in [(H // 4, W // 4), (H // 2, W // 2), (3 * H // 4, 3 * W // 4), (H // 4, 3 * W // 4)]:
        reg = im[cy - 128:cy + 128, cx - 128:cx + 128].astype(float)
        calib.append(("real", sigma_metric(reg), noise_metric(reg), np.percentile(reg, 99) - np.percentile(reg, 5)))
rng_c = np.random.default_rng(3)
for i in range(120):
    reg = make_sample(rng_c, augment=(i % 3 != 0), vacancy=0.0)["image"].astype(float)
    if (reg == 0).mean() > 0.02:
        continue
    calib.append(("synthetic", sigma_metric(reg), noise_metric(reg), np.percentile(reg, 99) - np.percentile(reg, 5)))
calib_df = pd.DataFrame(calib, columns=["source", "sigma_px", "noise", "contrast"])

print(calib_df.groupby("source").quantile([0.10, 0.25, 0.50, 0.75, 0.90]).round(3).unstack(0).to_string())

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(20, 8.4))

for j, name in enumerate(list(canon)[:5]):
    im = canon[name]; H, W = im.shape
    ax[0, j].imshow(im[H // 2 - 128:H // 2 + 128, W // 2 - 128:W // 2 + 128], cmap="gray", vmin=0, vmax=1)
    ax[0, j].set_title("real " + name[:22], fontsize=9); ax[0, j].axis("off")

rng_v = np.random.default_rng(5)
for j in range(5):
    s = make_sample(rng_v, augment=(j != 0))
    ax[1, j].imshow(s["image"], cmap="gray", vmin=0, vmax=1)
    ax[1, j].plot(s["heavy"][:, 0], s["heavy"][:, 1], "r.", ms=2); ax[1, j].plot(s["cu"][:, 0], s["cu"][:, 1], "c.", ms=2)
    ax[1, j].set_title(f"synthetic ({'augmented' if j else 'raw'}), Cu/heavy {s['meta']['cu_ratio']:.2f}", fontsize=9); ax[1, j].axis("off")

fig.suptitle("Top: real canonical crops.  Bottom: synthetic crops; red = heavy centres, cyan = Cu centres (from the label disks)")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "synthetic_vs_real.png", dpi=90, bbox_inches="tight"); plt.show()

### The network
Built on the *Shape Operator* architecture (analytic spatial kernels + learned 1×1 channel mixing, two-level U-Net,
parallel heads). Changes:

* **Blobs:** 8 elliptical Gaussians - learnable width (0.8–5.0 px, initialised
  1.5–4.0 px around the measured 2.75 px column σ) and axis ratio limited to 0.75–1 (near-circles to represent atoms) and angle. Mean-centered
  and L1-normalized, so each kernel is center minus surrounding. 21 × 21 window (2.9 σ at σ = 3.5 px).
* **Chain (Row) of Blobs:** ("columns of blobs" = lattice rows) - 6 learned mixes of the blob responses, each summed with
  copies of itself shifted by ±1 and ±2 learnable spacings along a learnable direction. Initial (spacing, angle) at
  16 px/period and c = 3.04 × 16 = 48.6 px: in-plane row (16 px, 0°), Ba–Y along c (0.31c = 15.1 px, 90°), Cu2–Cu2 along
  c (0.29c = 14.1 px, 90°), full c (48.6 px, 90°), and the heavy-lattice diagonals (√(16² + 15.1²) = 22.0 px, ±43°).
  Each chain is evaluated at θ and θ + 90° and the larger response is kept because the shared augmenter rotates by 90°.
* **Heads:** class logits plus, for each foreground class, the distance to the nearest column centre (divided by 8 px, clipped
  at 1). Columns are decoded as peaks of **P(class) × (1 − distance)**.

In [ ]:
def _grid(k):
    r = torch.arange(k, dtype=torch.float32) - k // 2
    y, x = torch.meshgrid(r, r, indexing="ij")
    return x.contiguous().clone(), y.contiguous().clone()   # meshgrid returns memory-sharing views; buffers must own memory


class BlobBank(nn.Module):
    def __init__(self, n_blobs=8, n_dog=0, k=21):
        super().__init__()
        self.k, self.n_dog = k, n_dog
        self.raw_s = nn.Parameter(torch.logit((torch.linspace(1.5, 4.0, n_blobs) - 0.8) / 4.2))
        self.raw_q = nn.Parameter(torch.zeros(n_blobs))
        self.angle = nn.Parameter(torch.arange(n_blobs) * math.pi / n_blobs)
        if n_dog:
            self.raw_d1 = nn.Parameter(torch.logit((torch.linspace(1.5, 3.5, n_dog) - 0.8) / 4.2))
            self.raw_dk = nn.Parameter(torch.zeros(n_dog))
        x, y = _grid(k); self.register_buffer("x", x); self.register_buffer("y", y)

    def params(self):
        return dict(sigma=(0.8 + 4.2 * torch.sigmoid(self.raw_s)).tolist(),
                    axis_ratio=(0.75 + 0.25 * torch.sigmoid(self.raw_q)).tolist(),
                    angle_deg=(torch.rad2deg(self.angle) % 180).tolist())

    def kernels(self):
        s = (0.8 + 4.2 * torch.sigmoid(self.raw_s))[:, None, None]
        q = (0.75 + 0.25 * torch.sigmoid(self.raw_q))[:, None, None]
        c, sn = self.angle.cos()[:, None, None], self.angle.sin()[:, None, None]
        u = c * self.x + sn * self.y; v = -sn * self.x + c * self.y
        ks = [torch.exp(-(u ** 2) / (2 * s ** 2) - (v ** 2) / (2 * (q * s) ** 2))]
        if self.n_dog:
            s1 = (0.8 + 4.2 * torch.sigmoid(self.raw_d1))[:, None, None]
            s2 = s1 * (1.2 + 1.8 * torch.sigmoid(self.raw_dk))[:, None, None]
            r2 = self.x ** 2 + self.y ** 2
            g1 = torch.exp(-r2 / (2 * s1 ** 2)); g2 = torch.exp(-r2 / (2 * s2 ** 2))
            ks.append(g1 / g1.sum((1, 2), keepdim=True) - g2 / g2.sum((1, 2), keepdim=True))
        k = torch.cat(ks); k = k - k.mean((1, 2), keepdim=True)
        return k / (k.abs().sum((1, 2), keepdim=True) + 1e-6)

    def forward(self, x):
        p = self.k // 2
        return F.conv2d(F.pad(x, (p, p, p, p), mode="reflect"), self.kernels()[:, None])


class ChainBank(nn.Module):
    INIT = [(16.0, 0.0), (15.1, 90.0), (14.1, 90.0), (48.6, 90.0), (22.0, 43.0), (22.0, -43.0)]

    def __init__(self, cin):
        super().__init__()
        n = len(self.INIT)
        self.mix = nn.Conv2d(cin, n, 1)
        self.spacing = nn.Parameter(torch.tensor([s for s, _ in self.INIT]))
        self.theta = nn.Parameter(torch.tensor([math.radians(a) for _, a in self.INIT]))
        self.raw_w = nn.Parameter(torch.zeros(n, 2))

    def params(self):
        return dict(spacing_px=self.spacing.tolist(), angle_deg=torch.rad2deg(self.theta).tolist(),
                    w1_w2=torch.sigmoid(self.raw_w).tolist())

    def _chain(self, r, theta):
        B, C, H, W = r.shape; w = torch.sigmoid(self.raw_w)
        gy, gx = torch.meshgrid(torch.linspace(-1, 1, H, device=r.device), torch.linspace(-1, 1, W, device=r.device), indexing="ij")
        rr = r.transpose(0, 1).reshape(C * B, 1, H, W); out = r
        for k in (-2, -1, 1, 2):
            dx = k * self.spacing * torch.cos(theta) * 2 / (W - 1); dy = k * self.spacing * torch.sin(theta) * 2 / (H - 1)
            grid = torch.stack([gx[None] + dx[:, None, None], gy[None] + dy[:, None, None]], -1)
            gg = grid[:, None].expand(C, B, H, W, 2).reshape(C * B, H, W, 2)
            sh = F.grid_sample(rr, gg, mode="bilinear", padding_mode="reflection", align_corners=True)
            out = out + w[:, abs(k) - 1].view(1, C, 1, 1) * sh.reshape(C, B, H, W).transpose(0, 1)
        return out

    def forward(self, blobs):
        r = self.mix(blobs)
        return torch.maximum(self._chain(r, self.theta), self._chain(r, self.theta + math.pi / 2))


class AnalyticBlock(nn.Module):
    def __init__(self, cin, cout, k=13):
        super().__init__()
        self.cin, self.k = cin, k
        self.scale = nn.Parameter(torch.zeros(4)); self.angle = nn.Parameter(torch.tensor(.2))
        x, y = _grid(k); self.register_buffer("x", x); self.register_buffer("y", y)
        self.mix = nn.Conv2d(cin * 4, cout, 1); self.norm = nn.GroupNorm(4, cout); self.res = nn.Conv2d(cin, cout, 1)

    def kernels(self):
        widths = .6 + 1.5 * self.scale.sigmoid(); ks = []
        u = self.angle.cos() * self.x + self.angle.sin() * self.y
        v = -self.angle.sin() * self.x + self.angle.cos() * self.y
        for j, w in enumerate(widths):
            g = torch.exp(-(self.x ** 2 + self.y ** 2) / (2 * w * w))
            if j == 0:
                k = g / g.sum()
            else:
                k = -u * g / w ** 2 if j == 1 else -v * g / w ** 2 if j == 2 else ((self.x ** 2 + self.y ** 2) / w ** 4 - 2 / w ** 2) * g
                k = k - k.mean(); k = k / (k.abs().sum() + 1e-6)
            ks.append(k)
        return torch.stack(ks)

    def forward(self, x):
        p = self.k // 2
        # The same 4 kernels are applied to every channel. Folding channels into the batch and running one dense 1 -> 4 convolution gives the grouped-convolution result (channel order c0k0..c0k3, c1k0, ...) and is much faster to backpropagate.
        B, C, H, W = x.shape
        h = F.conv2d(F.pad(x.reshape(B * C, 1, H, W), (p, p, p, p), mode="reflect"), self.kernels()[:, None])
        return F.silu(self.norm(self.mix(h.reshape(B, C * 4, H, W))) + self.res(x))


class ShapeNetYBCO(nn.Module):
    def __init__(self, n_classes, use_dog=False, n_blobs=8):
        super().__init__()
        self.n_classes = n_classes
        self.blob = BlobBank(n_blobs, n_dog=4 if use_dog else 0)
        nb = n_blobs + (4 if use_dog else 0)
        self.chain = ChainBank(nb + 1)
        self.e0 = AnalyticBlock(1 + nb + len(ChainBank.INIT), 12); self.e1 = AnalyticBlock(12, 20); self.e2 = AnalyticBlock(20, 28)
        self.b = AnalyticBlock(28, 28); self.d1 = AnalyticBlock(48, 20); self.d0 = AnalyticBlock(32, 12)
        self.class_head = nn.Conv2d(12, n_classes, 1); self.dist_head = nn.Conv2d(12, n_classes - 1, 1)

    def forward(self, x, features=False):
        f = {"blob": self.blob(x)}
        f["chain"] = self.chain(torch.cat([x, f["blob"]], 1))
        f["e0"] = self.e0(torch.cat([x, f["blob"], f["chain"]], 1))
        f["e1"] = self.e1(F.avg_pool2d(f["e0"], 2)); f["e2"] = self.e2(F.avg_pool2d(f["e1"], 2)); f["b"] = self.b(f["e2"])
        f["d1"] = self.d1(torch.cat([F.interpolate(f["b"], size=f["e1"].shape[-2:], mode="bilinear", align_corners=False), f["e1"]], 1))
        f["d0"] = self.d0(torch.cat([F.interpolate(f["d1"], size=f["e0"].shape[-2:], mode="bilinear", align_corners=False), f["e0"]], 1))
        logits = self.class_head(f["d0"]); dist = torch.sigmoid(self.dist_head(f["d0"]))
        return (logits, dist, f) if features else (logits, dist)


def count_parameters(model):
    analytic = sum(p.numel() for n, p in model.named_parameters()
                   if n.startswith("blob.") or n in ("chain.spacing", "chain.theta", "chain.raw_w") or n.endswith(".scale") or n.endswith(".angle"))
    return analytic, sum(p.numel() for p in model.parameters())


for nc, dog in [(2, False), (3, False), (3, True)]:
    a, t = count_parameters(ShapeNetYBCO(nc, use_dog=dog))
    print(f"{nc} classes, DoG={dog}: {t} parameters, of which {a} are analytic shape parameters")

### Targets, loss, decoding, and matching
* **Class:**

**V1:** heavy = 1, Cu disks = ignore.

**V2a:** heavy = 1, Cu = 2.

**V2b:** Cu = 1, heavy disks = background
  (heavy sites are not Cu, they are Y and Ba combined). Ignore pixels (augmenter padding and rotation corners) are excluded everywhere.

* **Loss** = class-weighted cross-entropy + L1 on the distance maps. Class weights are square-root-tempered inverse
  frequencies with background = 1 (used from my YBCO AtomAI U-Net notebook) and computed on each version's synthetic training set. The two terms
  are weighted equally [choice].
* **Threshold:** The detection threshold is picked by F1 on the synthetic validation set and frozen; every
  metric below is reported at that frozen threshold and the threshold is printed alongside it.
* **Matching:** 1-to-1 assignment within 4 px (tolerance chosen from my YBCO AtomAI U-Net notebook). Columns clipped by the crop border or the
  ignore region are neither credited nor penalized due to inconsistency.

In [ ]:
VERSIONS = {"V1_heavy": ["heavy"], "V2a_cu_direct": ["heavy", "Cu"], "V2b_cu_residual": ["Cu"]}

def class_target(s, version):
    lab = s["label"]; t = np.zeros(lab.shape, np.int64)
    if version == "V1_heavy":
        t[lab == HEAVY] = 1; t[lab == CU] = 255
    elif version == "V2a_cu_direct":
        t[lab == HEAVY] = 1; t[lab == CU] = 2
    else:
        t[lab == CU] = 1
    t[lab == IGNORE_CODE] = 255
    return t


def distance_target(s, version, shape):
    out = [s["dist"]["heavy" if cl == "heavy" else "cu"].astype(np.float32) for cl in VERSIONS[version]]
    mask = s["label"] != IGNORE_CODE
    if version == "V1_heavy":
        mask &= s["label"] != CU
    return np.stack(out), mask

class SynthDataset(torch.utils.data.Dataset):
    def __init__(self, samples, version, inputs=None):
        self.s, self.v, self.inputs = samples, version, inputs
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        s = self.s[i]; x = s["image"] if self.inputs is None else self.inputs[i]
        dt, dm = distance_target(s, self.v, x.shape)
        return (torch.from_numpy(np.asarray(x, np.float32))[None], torch.from_numpy(class_target(s, self.v)),
                torch.from_numpy(dt), torch.from_numpy(dm))

def class_weights(samples, version):
    n = len(VERSIONS[version]) + 1; counts = np.zeros(n)
    for s in samples:
        t = class_target(s, version); counts += np.bincount(t[t != 255].ravel(), minlength=n)[:n]
    w = np.sqrt(counts[0] / np.maximum(counts, 1)); return (w / w[0]).astype(np.float32)


def loss_fn(logits, dist, ct, dt, dm, weights):
    ce = F.cross_entropy(logits, ct, weight=weights, ignore_index=255)
    m = dm[:, None].expand_as(dist).float()
    return ce + (torch.abs(dist - dt) * m).sum() / m.sum().clamp_min(1)


def train(version, tr_s, va_s, epochs=EPOCHS, use_dog=False, tr_in=None, va_in=None, seed=17):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ShapeNetYBCO(len(VERSIONS[version]) + 1, use_dog=use_dog).to(DEVICE)
    w = torch.from_numpy(class_weights(tr_s, version)).to(DEVICE)
    tr = torch.utils.data.DataLoader(SynthDataset(tr_s, version, tr_in), batch_size=BATCH_SIZE, shuffle=True,
                                     generator=torch.Generator().manual_seed(seed))
    va = torch.utils.data.DataLoader(SynthDataset(va_s, version, va_in), batch_size=BATCH_SIZE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE); best, hist, t0 = float("inf"), [], time.time()
    for ep in range(epochs):
        model.train(); tl = 0.0; te = time.time(); nb = len(tr); report = max(1, nb // 10)
        for bi, (x, ct, dt, dm) in enumerate(tr, start=1):
            x, ct, dt, dm = x.to(DEVICE), ct.to(DEVICE), dt.to(DEVICE), dm.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(*model(x), ct, dt, dm, w); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5); opt.step(); tl += loss.item() * len(x)
            if ep == 0 and (bi == 5 or bi % report == 0):       # progress and time estimate during the first epoch
                per = (time.time() - te) / bi
                print(f"  epoch 1: batch {bi}/{nb}, {per:.2f} s/batch, about {per * nb / 60:.1f} min/epoch, "
                      f"{per * nb * epochs / 60:.0f} min for {epochs} epochs (validation not included)", flush=True)
        model.eval(); vl = 0.0
        with torch.no_grad():
            for x, ct, dt, dm in va:
                x, ct, dt, dm = x.to(DEVICE), ct.to(DEVICE), dt.to(DEVICE), dm.to(DEVICE)
                vl += loss_fn(*model(x), ct, dt, dm, w).item() * len(x)
        hist.append((ep + 1, tl / len(tr_s), vl / len(va_s)))
        if hist[-1][2] < best:
            best, state, best_ep = hist[-1][2], copy.deepcopy(model.state_dict()), ep + 1
        print(f"{version}{' +DoG' if use_dog else ''} epoch {ep + 1}: train {hist[-1][1]:.4f}  val {hist[-1][2]:.4f}  ({time.time() - t0:.0f} s)", flush=True)
    model.load_state_dict(state)
    return model, dict(history=hist, best_epoch=best_ep, class_weights=w.cpu().tolist())


@torch.no_grad()
def predict(model, images, batch=BATCH_SIZE):
    model.eval(); P, D = [], []
    for i in range(0, len(images), batch):
        x = torch.from_numpy(np.stack(images[i:i + batch]).astype(np.float32))[:, None].to(DEVICE)
        lg, ds = model(x); P.append(torch.softmax(lg, 1).cpu().numpy()); D.append(ds.cpu().numpy())
    return np.concatenate(P), np.concatenate(D)


@torch.no_grad()
def predict_large(model, img, tile=512, margin=48):
    model.eval(); H, W = img.shape; C = model.n_classes
    P = np.zeros((C, H, W), np.float32); D = np.zeros((C - 1, H, W), np.float32)
    pad = np.pad(img, margin, mode="reflect"); step = tile - 2 * margin
    for y in range(0, H, step):
        for x in range(0, W, step):
            y1, x1 = min(y + step, H), min(x + step, W)
            t = pad[y:y1 + 2 * margin, x:x1 + 2 * margin]
            t = np.pad(t, ((0, (-t.shape[0]) % 4), (0, (-t.shape[1]) % 4)), mode="reflect")
            lg, ds = model(torch.from_numpy(t.astype(np.float32))[None, None].to(DEVICE))
            P[:, y:y1, x:x1] = torch.softmax(lg, 1)[0].cpu().numpy()[:, margin:margin + y1 - y, margin:margin + x1 - x]
            D[:, y:y1, x:x1] = ds[0].cpu().numpy()[:, margin:margin + y1 - y, margin:margin + x1 - x]
    return P, D


def decode(prob, dist, threshold, min_dist=MIN_PEAK_DIST):
    """Local maxima of score = P(class) * (1 - distance), refined by a 3-point parabola along x and y.
    (A 3x3 weighted centroid was tested first: it snapped positions to whole pixels, mean fractional part 0.016.)"""
    score = prob * (1.0 - dist); r = int(np.ceil(min_dist))
    ys, xs = np.nonzero((score == maximum_filter(score, size=2 * r + 1)) & (score > threshold)); H, W = score.shape; out = []
    for y, x in zip(ys, xs):
        dx = dy = 0.0
        if 0 < x < W - 1:
            a, b, c = score[y, x - 1], score[y, x], score[y, x + 1]; den = a - 2 * b + c
            dx = 0.5 * (a - c) / den if den < 0 else 0.0
        if 0 < y < H - 1:
            a, b, c = score[y - 1, x], score[y, x], score[y + 1, x]; den = a - 2 * b + c
            dy = 0.5 * (a - c) / den if den < 0 else 0.0
        out.append((x + np.clip(dx, -0.5, 0.5), y + np.clip(dy, -0.5, 0.5), score[y, x]))
    return np.array(out, dtype=float).reshape(-1, 3)


def match(pred, truth, tol=MATCH_TOL):
    if len(pred) == 0 or len(truth) == 0:
        return 0, len(pred), len(truth), np.zeros(0)
    d = np.hypot(pred[:, None, 0] - truth[None, :, 0], pred[:, None, 1] - truth[None, :, 1])
    cost = np.where(d <= tol, d, 1e6); r, c = linear_sum_assignment(cost); ok = cost[r, c] < 1e6
    return int(ok.sum()), len(pred) - int(ok.sum()), len(truth) - int(ok.sum()), d[r[ok], c[ok]]


def evaluable(pts, s):
    if len(pts) == 0:
        return pts
    lab = s["label"]; H, W = lab.shape
    keep = lab[np.clip(np.round(pts[:, 1]).astype(int), 0, H - 1), np.clip(np.round(pts[:, 0]).astype(int), 0, W - 1)] != IGNORE_CODE
    part = np.r_[s["heavy_partial"], s["cu_partial"], s["cu_vacant_partial"]]
    if len(part):
        keep &= cKDTree(part).query(pts[:, :2])[0] > MATCH_TOL
    return pts[keep]


def evaluate(P, D, samples, version, cl, threshold, tol=MATCH_TOL):
    ci = VERSIONS[version].index(cl); key = "heavy" if cl == "heavy" else "cu"; tot = np.zeros(3); dd = []
    for i, s in enumerate(samples):
        pts = evaluable(decode(P[i, ci + 1], D[i, ci], threshold), s)
        tp, fp, fn, d = match(pts[:, :2], s[key], tol); tot += (tp, fp, fn); dd.append(d)
    tp, fp, fn = tot; pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    dd = np.concatenate(dd)
    return dict(threshold=threshold, precision=float(pr), recall=float(rc), f1=float(2 * pr * rc / max(pr + rc, 1e-9)),
                rmse_px=float(np.sqrt(np.mean(dd ** 2))) if len(dd) else np.nan, tp=int(tp), fp=int(fp), fn=int(fn))


def choose_threshold(P, D, samples, version, cl):
    scores = [(evaluate(P, D, samples, version, cl, t)["f1"], -abs(t - 0.5), t) for t in THRESHOLDS]
    t = max(scores)[2]
    if max(scores)[0] == 0:
        print(f"WARNING: {version}/{cl} has F1 = 0 at every threshold - the model detects nothing usable; "
              "the threshold defaults to the middle of the grid.")
    elif t in (min(THRESHOLDS), max(THRESHOLDS)):
        print(f"WARNING: {version}/{cl} threshold {t} is at the edge of THRESHOLDS - the optimum is not bracketed; "
              "extend the grid before trusting this operating point.")
    return t

### Synthetic datasets (augmented datasets)
Separate seeds for training, validation, test, and the detection-limit sweep. Distance maps are stored as f16 (float16) with
each crop (about 0.8 GB per 3,000 crops). V2b gets its own train/validation/test
sets: V1 is run on them to build the residuals and V1 has never seen them before, so V2b trains on residuals with the same
kind of V1 errors it will encounter on real images.

In [ ]:
t0 = time.time()

train_s = make_dataset(N_TRAIN, SEEDS["train"]); val_s = make_dataset(N_VAL, SEEDS["val"]); test_s = make_dataset(N_TEST, SEEDS["test"])
sweep_s = {r: make_dataset(N_PER_RATIO, SEEDS["sweep"] + int(round(100 * r)), cu_ratio=r) for r in SWEEP_RATIOS}

print(f"generated {N_TRAIN} + {N_VAL} + {N_TEST} + {len(SWEEP_RATIOS)} x {N_PER_RATIO} crops in {time.time() - t0:.0f} s")
print("mean heavy / Cu centres per training crop: %.1f / %.1f" % (np.mean([len(s['heavy']) for s in train_s]), np.mean([len(s['cu']) for s in train_s])))

---
## Version 1 — Heavy columns (Y and Ba as one heavy column class)

In [ ]:
v1, v1_log = train("V1_heavy", train_s, val_s)

P, D = predict(v1, [s["image"] for s in val_s]); T_V1 = choose_threshold(P, D, val_s, "V1_heavy", "heavy")
P, D = predict(v1, [s["image"] for s in test_s])
v1_test = evaluate(P, D, test_s, "V1_heavy", "heavy", T_V1)

print(f"best epoch {v1_log['best_epoch']}, class weights {np.round(v1_log['class_weights'], 3)}")
print("V1 heavy, synthetic test:", {k: round(v, 4) if isinstance(v, float) else v for k, v in v1_test.items()})
print("tolerance sweep (F1):", {t: round(evaluate(P, D, test_s, "V1_heavy", "heavy", T_V1, t * CANON_PERIOD)["f1"], 4) for t in TOLERANCE_SWEEP_REL})

torch.save(v1.state_dict(), OUTPUT_DIR / "V1_heavy.pt")

---
## Version 2a — Cu found directly on the HAADF image
**Three classes** (background, heavy, Cu) from the raw HAADF image. This is the **independent** Cu test.

In [ ]:
v2a, v2a_log = train("V2a_cu_direct", train_s, val_s)

P, D = predict(v2a, [s["image"] for s in val_s])
T_V2A = {cl: choose_threshold(P, D, val_s, "V2a_cu_direct", cl) for cl in ["heavy", "Cu"]}
P, D = predict(v2a, [s["image"] for s in test_s])
v2a_test = {cl: evaluate(P, D, test_s, "V2a_cu_direct", cl, T_V2A[cl]) for cl in ["heavy", "Cu"]}

print(f"best epoch {v2a_log['best_epoch']}, class weights {np.round(v2a_log['class_weights'], 3)}")

for cl, r in v2a_test.items():
    print(f"V2a {cl}, synthetic test:", {k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()})
print("Cu tolerance sweep (F1):", {t: round(evaluate(P, D, test_s, "V2a_cu_direct", "Cu", T_V2A["Cu"], t * CANON_PERIOD)["f1"], 4) for t in TOLERANCE_SWEEP_REL})

torch.save(v2a.state_dict(), OUTPUT_DIR / "V2a_cu_direct.pt")

---
## Version 2b — Cu found on the residual after subtracting heavy columns
V1 finds heavy columns; each is modeled as a Gaussian (width = median of free 2D fits on up to 200 detected columns
per real image and 30 per synthetic crop;
amplitude = 3×3 mean at the column minus the 10th % within one period) and subtracted. A two-class model
(background, Cu) is trained on residuals and built the same way from fresh synthetic sets.

**For the later comparison with my AtomAI version (it would be interesting to test this more directly at some point):** The AtomAI Cu labels also came from a residual, so agreement between V2b and
those labels is partly built in. V2a is the independent test.

In [ ]:
def fit_sigma(img, pts, n=200, seed=0):
    rng = np.random.default_rng(seed); sel = pts if len(pts) <= n else pts[rng.choice(len(pts), n, replace=False)]
    H, W = img.shape; out = []
    for x, y in sel[:, :2]:
        xi, yi = int(round(x)), int(round(y))
        if 6 <= xi <= W - 7 and 6 <= yi <= H - 7:
            out.append(gauss_fit_sigma(img[yi - 5:yi + 6, xi - 5:xi + 6].astype(float), x - xi, y - yi))
    return float(np.median(out)) if out else float("nan")


def subtract_heavy(img, heavy_pts, sigma):
    img = img.astype(np.float32); H, W = img.shape; model = np.zeros_like(img)
    w = int(np.ceil(4 * sigma)); half = int(CANON_PERIOD // 2)
    for x, y in heavy_pts[:, :2]:
        xi, yi = int(round(x)), int(round(y))
        if not (0 <= xi < W and 0 <= yi < H):
            continue
        A = max(img[max(0, yi - 1):yi + 2, max(0, xi - 1):xi + 2].mean()
                - np.percentile(img[max(0, yi - half):yi + half + 1, max(0, xi - half):xi + half + 1], 10), 0.0)
        x0, x1, y0, y1 = max(0, xi - w), min(W, xi + w + 1), max(0, yi - w), min(H, yi + w + 1)
        model[y0:y1, x0:x1] += A * np.outer(np.exp(-((np.arange(y0, y1) - y) ** 2) / (2 * sigma ** 2)),
                                            np.exp(-((np.arange(x0, x1) - x) ** 2) / (2 * sigma ** 2)))
    return img - model


def residuals(samples):
    P, D = predict(v1, [s["image"] for s in samples]); out = []
    for i, s in enumerate(samples):
        pts = decode(P[i, 1], D[i, 0], T_V1); sig = fit_sigma(s["image"], pts, n=30)   # 30 fits per crop; 200 on real images
        out.append(subtract_heavy(s["image"], pts, sig if np.isfinite(sig) else 2.75))
    return out


res_train_s = make_dataset(N_TRAIN, SEEDS["res_train"]); res_val_s = make_dataset(N_VAL, SEEDS["res_val"]); res_test_s = make_dataset(N_TEST, SEEDS["res_test"])
res_train_in, res_val_in, res_test_in = residuals(res_train_s), residuals(res_val_s), residuals(res_test_s)
v2b, v2b_log = train("V2b_cu_residual", res_train_s, res_val_s, tr_in=res_train_in, va_in=res_val_in)
P, D = predict(v2b, res_val_in); T_V2B = choose_threshold(P, D, res_val_s, "V2b_cu_residual", "Cu")
P, D = predict(v2b, res_test_in); v2b_test = evaluate(P, D, res_test_s, "V2b_cu_residual", "Cu", T_V2B)

print(f"best epoch {v2b_log['best_epoch']}, class weights {np.round(v2b_log['class_weights'], 3)}")
print("V2b Cu, synthetic test:", {k: round(v, 4) if isinstance(v, float) else v for k, v in v2b_test.items()})
print("Cu tolerance sweep (F1):", {t: round(evaluate(P, D, res_test_s, "V2b_cu_residual", "Cu", T_V2B, t * CANON_PERIOD)["f1"], 4) for t in TOLERANCE_SWEEP_REL})

torch.save(v2b.state_dict(), OUTPUT_DIR / "V2b_cu_residual.pt")

### Shape-network diagnostics
For every trained model: **predictions** (Input | True class | Pred class | true and predicted geometry on the true class
support), **filters** (initial kernels | learned kernels | response to a test image, for the blob bank and the chain
bank), **activations** (first six channels at every stage), **analytic kernels** (the four operators in every block),
**training curves**, and **real-image probes** (raw outputs everywhere, no ground truth).

* **"Geometry"** here is the distance from each pixel to its column center in px; it is shown only on true class support,
  so a missed column still shows its true value. Class colors: 0 = background, 1 = heavy (Cu in V2b), 2 = Cu; white =
  ignored pixels.
* Panels show a 96×96 px window of each 256×256 crop (6 in-plane periods, about 2 unit cells along c) so individual
  columns are visible.
* Initial kernels are deterministic (no random initialization in the blob, chain or analytic parameters), so they are
  rebuilt from a fresh model.
* Kernel and activation panels use independent symmetric color scales; compare shapes and not magnitudes across panels.

In [ ]:
DIAG_DIR = OUTPUT_DIR / "diagnostics"; DIAG_DIR.mkdir(exist_ok=True)
WIN = 96                                           # [choice] display window, px
CLASS_LABEL = {"V1_heavy": "0=background, 1=heavy", "V2a_cu_direct": "0=background, 1=heavy, 2=Cu",
               "V2b_cu_residual": "0=background, 1=Cu"}
MODELS = {"V1_heavy": (v1, "V1_heavy", None, test_s, None, v1_log),
          "V2a_cu_direct": (v2a, "V2a_cu_direct", None, test_s, None, v2a_log),
          "V2b_cu_residual": (v2b, "V2b_cu_residual", None, res_test_s, res_test_in, v2b_log)}


def _win(a, y0=(CROP - WIN) // 2, x0=(CROP - WIN) // 2, f=1):
    return a[..., y0 // f:(y0 + WIN) // f, x0 // f:(x0 + WIN) // f]


def _sym(ax, v, title=None):
    lim = max(float(np.nanmax(np.abs(v))), 1e-6); ax.imshow(v, cmap="RdBu_r", vmin=-lim, vmax=lim); ax.axis("off")
    if title: ax.set_title(title, fontsize=8)


def plot_predictions(model, version, samples, inputs, path, title, n=3):
    imgs = [s["image"] for s in samples[:n]] if inputs is None else list(inputs[:n])
    P, D = predict(model, imgs); pred = P.argmax(1); classes = VERSIONS[version]
    titles = ["Input", "True class", "Pred class"] + [t for cl in classes for t in (f"True dist {cl} / px", f"Pred dist {cl} / px")]
    fig, ax = plt.subplots(n, len(titles), figsize=(2.3 * len(titles), 2.6 * n), constrained_layout=True, squeeze=False)
    rmax = ATOM_RADIUS_REL * CANON_PERIOD
    for i in range(n):
        ct = class_target(samples[i], version); ctf = ct.astype(float); ctf[ct == 255] = np.nan
        vals = [imgs[i], ctf, pred[i].astype(float)]
        for ci, cl in enumerate(classes):
            sup = ct == ci + 1; td = samples[i]["dist"]["heavy" if cl == "heavy" else "cu"].astype(float) * DIST_SCALE
            vals += [np.where(sup, td, np.nan), np.where(sup, D[i, ci] * DIST_SCALE, np.nan)]
        for j, v in enumerate(vals):
            a = ax[i, j]; v = _win(v)
            if j == 0:
                a.imshow(v, cmap="gray")
            elif j in (1, 2):
                a.imshow(v, cmap="tab10", vmin=0, vmax=2)
            else:
                a.imshow(v, cmap="viridis", vmin=0, vmax=rmax)
            a.axis("off")
            if i == 0:
                a.set_title(titles[j], fontsize=9)
    fig.suptitle(f"{title} | geometry on true class support; class {CLASS_LABEL[version]} | {WIN}x{WIN} px window")
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)


def _chain_canvas(spacing, theta, w, size):
    yy, xx = np.mgrid[:size, :size] - size // 2; img = np.zeros((size, size))
    for th in (theta, theta + np.pi / 2):
        for k, wk in zip((-2, -1, 0, 1, 2), (w[1], w[0], 1.0, w[0], w[1])):
            cx, cy = k * spacing * np.cos(th), k * spacing * np.sin(th)
            img += wk * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 2.5 ** 2))
    return img


@torch.no_grad()
def plot_filters(model, use_dog, x, path, title):
    model.eval(); fresh = ShapeNetYBCO(model.n_classes, use_dog=bool(use_dog))
    xin = torch.from_numpy(np.asarray(x, np.float32))[None, None].to(DEVICE)
    _, _, f = model(xin, features=True)
    k0, k1 = fresh.blob.kernels().numpy(), model.blob.kernels().cpu().numpy(); bp = model.blob.params()
    n = len(k0); fig, ax = plt.subplots(3, n, figsize=(1.9 * n, 6), constrained_layout=True)
    for j in range(n):
        _sym(ax[0, j], k0[j], f"blob {j + 1}" if j < len(bp["sigma"]) else f"DoG {j + 1 - len(bp['sigma'])}")
        lab = (f"σ {bp['sigma'][j]:.2f}, q {bp['axis_ratio'][j]:.2f}" if j < len(bp["sigma"]) else "learned")
        _sym(ax[1, j], k1[j], lab); _sym(ax[2, j], _win(f["blob"][0, j].cpu().numpy()))
    for i, lab in enumerate(["Initial kernels", "Learned kernels", "Response"]):
        ax[i, 0].text(-0.15, 0.5, lab, transform=ax[i, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
    fig.suptitle(title + " — BLOB BANK: initial | learned | response (independent colour scales)")
    fig.savefig(path / "filters_blob.png", dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
    c0, c1 = fresh.chain.params(), model.chain.params(); m = len(c1["spacing_px"])
    size = 2 * int(np.ceil(2 * max(abs(np.r_[c0["spacing_px"], c1["spacing_px"]])) + 8)) + 1   # every +/-2 tap fits
    fig, ax = plt.subplots(3, m, figsize=(2.3 * m, 6.6), constrained_layout=True)
    for j in range(m):
        for i, cp in enumerate([c0, c1]):
            ax[i, j].imshow(_chain_canvas(cp["spacing_px"][j], np.radians(cp["angle_deg"][j]), cp["w1_w2"][j], size), cmap="magma"); ax[i, j].axis("off")
            ax[i, j].set_title(f"s {cp['spacing_px'][j]:.1f} px, θ {cp['angle_deg'][j]:.1f}°\nw {cp['w1_w2'][j][0]:.2f}, {cp['w1_w2'][j][1]:.2f}", fontsize=8)
        _sym(ax[2, j], _win(f["chain"][0, j].cpu().numpy()))
    for i, lab in enumerate(["Initial taps", "Learned taps", "Response"]):
        ax[i, 0].text(-0.15, 0.5, lab, transform=ax[i, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
    fig.suptitle(title + f" — CHAIN BANK (rows of blobs): tap positions at θ and θ+90° ({size} px canvas), and response")
    fig.savefig(path / "filters_chain.png", dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)
    keys = ["blob", "chain", "e0", "e1", "e2", "b", "d1", "d0"]; fac = dict(blob=1, chain=1, e0=1, e1=2, e2=4, b=4, d1=2, d0=1)
    fig, ax = plt.subplots(len(keys), 6, figsize=(12, 2.0 * len(keys)), constrained_layout=True)
    for i, key in enumerate(keys):
        arr = f[key][0].cpu().numpy()
        for j in range(6):
            _sym(ax[i, j], _win(arr[j], f=fac[key]), f"{key}, ch {j}, {arr.shape[1]}x{arr.shape[2]}")
    fig.suptitle(title + " — first six channels at each stage (window shown; own symmetric colour scale per map)")
    fig.savefig(path / "activations.png", dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)
    blocks = ["e0", "e1", "e2", "b", "d1", "d0"]; fig, ax = plt.subplots(6, 4, figsize=(8, 12), constrained_layout=True)
    for i, key in enumerate(blocks):
        blk = getattr(model, key); wd = (.6 + 1.5 * blk.scale.sigmoid()).cpu().numpy()
        for j, k in enumerate(blk.kernels().cpu().numpy()):
            _sym(ax[i, j], k, f"{key}: {['Gaussian', 'derivative u', 'derivative v', 'LoG'][j]} w={wd[j]:.2f}")
    fig.suptitle(title + " — learned analytic kernels (13x13)")
    fig.savefig(path / "analytic_kernels.png", dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)


def plot_training(logs, path):
    fig, ax = plt.subplots(1, len(logs), figsize=(4.2 * len(logs), 3.4), constrained_layout=True, squeeze=False)
    for a, (name, lg) in zip(ax[0], logs.items()):
        h = np.array(lg["history"]); a.plot(h[:, 0], h[:, 1], label="train"); a.plot(h[:, 0], h[:, 2], label="validation")
        a.axvline(lg["best_epoch"], color="0.6", ls="--", label=f"kept: epoch {lg['best_epoch']}")
        a.set_title(name, fontsize=10); a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(fontsize=8)
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.show(); plt.close(fig)


for name, (model, version, dog, samples, inputs, lg) in MODELS.items():
    path = DIAG_DIR / name; path.mkdir(exist_ok=True)
    plot_predictions(model, version, samples, inputs, path / "predictions.png", f"{name} trained and tested (synthetic test set)")
    plot_filters(model, dog, samples[0]["image"] if inputs is None else inputs[0], path, name)

plot_training({k: v[5] for k, v in MODELS.items()}, DIAG_DIR / "training.png")

In [ ]:
# Real-image probes: three images spanning the magnification range; raw outputs everywhere (no ground truth).
@torch.no_grad()
def probe_real(name, crop=256):
    im = canon[name]; H, W = im.shape; y0, x0 = H // 2 - crop // 2, W // 2 - crop // 2; x = im[y0:y0 + crop, x0:x0 + crop]
    P1, D1 = predict(v1, [x]); P2, D2 = predict(v2a, [x])
    heavy = decode(P1[0, 1], D1[0, 0], T_V1); sig = fit_sigma(x, heavy, n=30)
    r = subtract_heavy(x, heavy, sig if np.isfinite(sig) else 2.75); P3, D3 = predict(v2b, [r])
    vals = [(x, "Input", "gray", None), (P1[0].argmax(0), "V1 class", "tab10", 2), (P1[0, 1], "V1 P(heavy)", "viridis", 1),
            (D1[0, 0] * DIST_SCALE, "V1 dist heavy / px", "viridis", DIST_SCALE),
            (P2[0].argmax(0), "V2a class", "tab10", 2), (P2[0, 2], "V2a P(Cu)", "viridis", 1), (D2[0, 1] * DIST_SCALE, "V2a dist Cu / px", "viridis", DIST_SCALE),
            (r, "V2b input: residual", "gray", None), (P3[0, 1], "V2b P(Cu)", "viridis", 1), (D3[0, 0] * DIST_SCALE, "V2b dist Cu / px", "viridis", DIST_SCALE)]
    fig, ax = plt.subplots(2, 5, figsize=(17, 7.4), constrained_layout=True)
    for a, (v, t, cm, vmax) in zip(ax.flat, vals):
        o = a.imshow(_win(v), cmap=cm, **({} if vmax is None else dict(vmin=0, vmax=vmax))); a.set_title(t, fontsize=10); a.axis("off")
        if cm == "viridis":
            fig.colorbar(o, ax=a, shrink=.7)
    fig.suptitle(f"Real image {name} (canonical scale, {WIN}x{WIN} px window of the centre crop) — raw outputs shown everywhere; "
                 "values outside columns are unconstrained")
    fig.savefig(DIAG_DIR / f"real_probe_{name[:4]}.png", dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)

for pref in ["1644", "1117", "1739"]:            # 4.10 Mx, 2.90 Mx, 1.45 Mx
    probe_real([n for n in canon if n.startswith(pref)][0])

---
## Detection limit: Cu F1 against Cu/heavy brightness
Each point is 40 synthetic crops rendered at one Cu/heavy ratio, scored at each model's frozen threshold. The
gray band is the interquartile range of Cu/heavy measured in the real average cells.

In [ ]:
def sweep_f1(model, version, cl, threshold, residual=False):
    out = []
    for r in SWEEP_RATIOS:
        ss = sweep_s[r]; imgs = residuals(ss) if residual else [s["image"] for s in ss]
        P, D = predict(model, imgs); out.append(evaluate(P, D, ss, version, cl, threshold))
    return out

curves = {"V2a Cu direct": sweep_f1(v2a, "V2a_cu_direct", "Cu", T_V2A["Cu"]),
          "V2b Cu residual": sweep_f1(v2b, "V2b_cu_residual", "Cu", T_V2B, residual=True),
          "V1 heavy": sweep_f1(v1, "V1_heavy", "heavy", T_V1)}

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
iq = kept_clean.Cu_heavy.quantile([.25, .75]).values

for a, key in zip(ax, ["f1", "recall", "precision"]):
    a.axvspan(*iq, color="0.85", label="measured Cu/heavy IQR")
    for name, res in curves.items():
        a.plot(SWEEP_RATIOS, [r[key] for r in res], "o-", label=name)
    a.set_xlabel("rendered Cu / heavy"); a.set_ylabel(key); a.set_ylim(0, 1.02); a.grid(alpha=.3)

ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(OUTPUT_DIR / "detection_limit.png", dpi=110, bbox_inches="tight"); plt.show()
print(pd.DataFrame({n: [round(r["f1"], 3) for r in res] for n, res in curves.items()}, index=SWEEP_RATIOS).to_string())

---
## Negative control: are empty Cu sites marked as Cu?
Two sets of synthetic crops at Cu/heavy = 0.52 (the measured median): **all Cu removed** and **half the Cu removed**.
For each Cu model at its frozen threshold:
* **vacant-site marking rate** = fraction of empty Cu sites with a Cu prediction within 4 px. A model that infers Cu
  from the heavy lattice marks empty sites about as often as it finds occupied ones; a model that uses Cu brightness
  marks few.
* **occupied-site recall** (half-removed set) = fraction of the remaining Cu columns found.

Empty sites are at a minimum of 10 px from any heavy or Cu column, so a 4 px match cannot be confused with a neighbour.

In [ ]:
def control_rates(model, version, samples, threshold, residual=False):
    imgs = residuals(samples) if residual else [s["image"] for s in samples]
    P, D = predict(model, imgs); ci = VERSIONS[version].index("Cu")
    n_vac = n_marked = tp = fn = n_pred = 0; preds = []
    for i, s in enumerate(samples):
        pts = evaluable(decode(P[i, ci + 1], D[i, ci], threshold), s)[:, :2]; preds.append(pts); n_pred += len(pts)
        if len(s["cu"]):
            t, _, f, _ = match(pts, s["cu"]); tp += t; fn += f
        vac = s["cu_vacant"]; n_vac += len(vac)
        if len(vac) and len(pts):
            n_marked += int((cKDTree(pts).query(vac)[0] <= MATCH_TOL).sum())
    return dict(vacant_sites=n_vac, vacant_marked=n_marked, vacant_marked_rate=n_marked / max(n_vac, 1),
                occupied_recall=(tp / (tp + fn)) if (tp + fn) else float("nan"), cu_predictions=n_pred), preds


ctrl_sets = {"all Cu removed": make_dataset(N_CTRL, SEEDS["ctrl_absent"], cu_ratio=CTRL_CU_RATIO, vacancy=1.0),
             "half Cu removed": make_dataset(N_CTRL, SEEDS["ctrl_half"], cu_ratio=CTRL_CU_RATIO, vacancy=0.5)}

ctrl_models = [("V2a Cu direct", v2a, "V2a_cu_direct", T_V2A["Cu"], False), ("V2b Cu residual", v2b, "V2b_cu_residual", T_V2B, True)]
rows, ctrl_preds = [], {}

for set_name, ss in ctrl_sets.items():
    for name, model, version, thr, resid in ctrl_models:
        r, preds = control_rates(model, version, ss, thr, resid); ctrl_preds[(set_name, name)] = preds
        rows.append(dict(set=set_name, model=name, threshold=thr, **r))
control_df = pd.DataFrame(rows)
control_df.to_csv(OUTPUT_DIR / "negative_control.csv", index=False)
print(control_df.round(3).to_string(index=False))

# green circles = Cu present, red x = empty Cu site, cyan + = Cu predicted
ss = ctrl_sets["half Cu removed"]; y0 = x0 = (CROP - WIN) // 2 if "WIN" in globals() else 80; win = WIN if "WIN" in globals() else 96
fig, ax = plt.subplots(len(ctrl_models), 3, figsize=(12, 4.1 * len(ctrl_models)), constrained_layout=True)

for r_, (name, model, version, thr, resid) in enumerate(ctrl_models):
    for c_ in range(3):
        s = ss[c_]; pts = ctrl_preds[("half Cu removed", name)][c_]; a = ax[r_, c_]
        img = residuals([s])[0] if resid else s["image"]
        a.imshow(img[y0:y0 + win, x0:x0 + win], cmap="gray", extent=(x0, x0 + win, y0 + win, y0))
        inw = lambda p: p[(p[:, 0] >= x0) & (p[:, 0] < x0 + win) & (p[:, 1] >= y0) & (p[:, 1] < y0 + win)] if len(p) else p.reshape(-1, 2)
        cu, vac, pr = inw(s["cu"]), inw(s["cu_vacant"]), inw(pts)
        a.scatter(cu[:, 0], cu[:, 1], s=60, facecolors="none", edgecolors="lime", linewidths=1.2)
        a.scatter(vac[:, 0], vac[:, 1], s=50, marker="x", color="red", linewidths=1.5)
        a.scatter(pr[:, 0], pr[:, 1], s=40, marker="+", color="cyan", linewidths=1.3)
        a.set_title(f"{name}, crop {c_ + 1}", fontsize=9); a.axis("off")

fig.suptitle("Negative control, half the Cu removed (Cu/heavy 0.52): green o = Cu present, red x = empty Cu site, cyan + = Cu predicted")
fig.savefig(OUTPUT_DIR / "negative_control.png", dpi=100, bbox_inches="tight"); plt.show(); plt.close(fig)

---
## Real images: inference on all 14, overlays, coordinates, and the Cu:heavy rastio check
* Models run on the canonical images (tiled, reflect-padded margins); coordinates are mapped back to native pixels with zoom-consistent mapping.
* **Counts:** Use columns at least 1 period inside the image border.
* **Expected Ratios (crystal):** Per projected cell there are 3 heavy (Ba, Y, Ba) and 3 Cu (Cu1, Cu2, Cu2 (explained decision below)), so
  **Cu : heavy = 1** and **Cu2 : Cu1 = 2**.
* **Cu1 vs Cu2:** Decided locally - the nearest heavy column above and below a Cu (along c, within half a period
  sideways) are found, and their separation is compared with 0.345 c — midway between the measured Ba–Y gaps (0.306 and 0.312 c)
  and Ba–Ba gap (0.381 c). Cu1 sits in the Ba–Ba gap.
* **Site geometry:** For every Cu, nearest-heavy distance ÷ median heavy nearest-neighbour distance. The measured average
  cells give ≈ 0.70 (Cu2) and ≈ 0.80 (Cu1).

In [ ]:
def interior(pts, shape, margin):
    H, W = shape
    return pts[(pts[:, 0] >= margin) & (pts[:, 0] <= W - 1 - margin) & (pts[:, 1] >= margin) & (pts[:, 1] <= H - 1 - margin)]


def cu_sites(cu, heavy, c_px):
    """Classify each Cu as Cu1/Cu2 from the heavy columns directly above and below it; also return site ratios."""
    tree = cKDTree(heavy[:, :2]); nn_hh = np.median(tree.query(heavy[:, :2], k=2)[0][:, 1])
    labels, ratios = [], tree.query(cu[:, :2])[0] / nn_hh
    for x, y in cu[:, :2]:
        idx = tree.query_ball_point((x, y), 0.8 * c_px)
        if not idx:
            labels.append("unclassified"); continue
        d = heavy[idx, :2] - (x, y)
        side = np.abs(np.abs(d[:, 0]) - CANON_PERIOD / 2) < CANON_PERIOD / 4      # heavy columns half a period sideways
        up, dn = d[side & (d[:, 1] < 0), 1], d[side & (d[:, 1] > 0), 1]
        if len(up) == 0 or len(dn) == 0:
            labels.append("unclassified"); continue
        labels.append("Cu1" if (dn.min() - up.max()) > 0.345 * c_px else "Cu2")
    return np.array(labels), ratios, nn_hh

real = {}

for name, im in canon.items():
    t0 = time.time(); r = scan_df.loc[name]; c_px = r.c_over_a * CANON_PERIOD
    P1, D1 = predict_large(v1, im); heavy_v1 = decode(P1[1], D1[0], T_V1)
    P2, D2 = predict_large(v2a, im); heavy_v2a = decode(P2[1], D2[0], T_V2A["heavy"]); cu_v2a = decode(P2[2], D2[1], T_V2A["Cu"])
    sig = fit_sigma(im, heavy_v1); resid = subtract_heavy(im, heavy_v1, sig)
    P3, D3 = predict_large(v2b, resid); cu_v2b = decode(P3[1], D3[0], T_V2B)
    out = dict(heavy_v1=heavy_v1, heavy_v2a=heavy_v2a, cu_v2a=cu_v2a, cu_v2b=cu_v2b, sigma=sig, residual=resid)
    m = CANON_PERIOD
    hv = interior(heavy_v1, im.shape, m)
    for tag, cu in [("v2a", cu_v2a), ("v2b", cu_v2b)]:
        ci = interior(cu, im.shape, m); lab, rat, nn_hh = cu_sites(ci, heavy_v1, c_px)
        out[f"stats_{tag}"] = dict(n_heavy=len(hv), n_cu=len(ci), cu_per_heavy=len(ci) / max(len(hv), 1),
                                   n_cu1=int((lab == "Cu1").sum()), n_cu2=int((lab == "Cu2").sum()),
                                   cu2_per_cu1=(lab == "Cu2").sum() / max((lab == "Cu1").sum(), 1),
                                   unclassified=int((lab == "unclassified").sum()),
                                   site_ratio_median=float(np.median(rat)) if len(rat) else np.nan,
                                   frac_site_below_0p6=float((rat < 0.6).mean()) if len(rat) else np.nan, heavy_nn_px=float(nn_hh))
        out[f"site_ratios_{tag}"] = rat
    real[name] = out

    # coordinates in native pixels
    ns, cs = r.native_shape, im.shape
    for key in ["heavy_v1", "heavy_v2a", "cu_v2a", "cu_v2b"]:
        nat = canonical_to_native(out[key][:, :2], ns, cs)
        pd.DataFrame(dict(x_px=nat[:, 0], y_px=nat[:, 1], score=out[key][:, 2])).to_csv(OUTPUT_DIR / f"{name}_{key}.csv", index=False)

    print(f"{name}: heavy {len(heavy_v1)} | Cu V2a {len(cu_v2a)} | Cu V2b {len(cu_v2b)} | sigma {sig:.2f} px | {time.time() - t0:.0f} s")

In [ ]:
rows = []

for name, o in real.items():
    for tag, lab in [("v2a", "V2a direct"), ("v2b", "V2b residual")]:
        s = o[f"stats_{tag}"]; rows.append(dict(image=name[:28], version=lab, **{k: (round(v, 3) if isinstance(v, float) else v) for k, v in s.items()}))

ratio_df = pd.DataFrame(rows)
print("Expected from the crystal structure: cu_per_heavy = 1.0, cu2_per_cu1 = 2.0, site_ratio ~0.70 (Cu2) / ~0.80 (Cu1)\n")
print(ratio_df.to_string(index=False))
ratio_df.to_csv(OUTPUT_DIR / "real_image_ratios.csv", index=False)
print("\nmedian over the 14 images:")
print(ratio_df.groupby("version")[["cu_per_heavy", "cu2_per_cu1", "site_ratio_median", "frac_site_below_0p6"]].median().round(3).to_string())

In [ ]:
# Overlays: full field and an 8 nm zoom; V2a and V2b are drawn in separate panels.
(OUTPUT_DIR / "overlays").mkdir(exist_ok=True)
for name, o in real.items():
    im = canon[name]; H, W = im.shape; r = scan_df.loc[name]
    z = int(round(80.0 / r.a_A * CANON_PERIOD)); cy, cx = H // 2, W // 2; y0, x0 = cy - z // 2, cx - z // 2
    fig, ax = plt.subplots(1, 4, figsize=(24, 6.4))
    ax[0].imshow(im, cmap="gray", vmin=0, vmax=1); ax[0].add_patch(plt.Rectangle((x0, y0), z, z, fill=False, ec="yellow"))
    ax[0].set_title(f"{name}\nfull field (yellow = zoom)")
    panels = [("V1 heavy", [(o["heavy_v1"], "red")]),
              ("V2a — Cu direct", [(o["heavy_v1"], "red"), (o["cu_v2a"], "cyan")]),
              ("V2b — Cu residual", [(o["heavy_v1"], "red"), (o["cu_v2b"], "yellow")])]
    for a, (title, layers) in zip(ax[1:], panels):
        a.imshow(im[y0:y0 + z, x0:x0 + z], cmap="gray", vmin=0, vmax=1, extent=(x0, x0 + z, y0 + z, y0))
        for pts, col in layers:
            sel = pts[(pts[:, 0] >= x0) & (pts[:, 0] < x0 + z) & (pts[:, 1] >= y0) & (pts[:, 1] < y0 + z)]
            a.scatter(sel[:, 0], sel[:, 1], s=14, facecolors="none", edgecolors=col, linewidths=1.0)
        st = o["stats_v2a"] if "V2a" in title else o["stats_v2b"] if "V2b" in title else None
        a.set_title(title + ("" if st is None else f"\nCu/heavy {st['cu_per_heavy']:.2f}, Cu2/Cu1 {st['cu2_per_cu1']:.2f}"))
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / "overlays" / f"{name}.png", dpi=80, bbox_inches="tight"); plt.show(); plt.close(fig)

In [ ]:
# Cu site geometry over all 14 images
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, tag, title in [(ax[0], "v2a", "V2a — Cu direct"), (ax[1], "v2b", "V2b — Cu residual")]:
    allr = np.concatenate([o[f"site_ratios_{tag}"] for o in real.values()])
    a.hist(allr, bins=np.linspace(0.2, 1.2, 51), color="0.4")
    for v, lab in [(0.5, "edge midpoint 0.5"), (0.70, "Cu2 0.70"), (0.80, "Cu1 0.80")]:
        a.axvline(v, ls="--", label=lab, color={0.5: "r", 0.70: "b", 0.80: "g"}[v])
    a.set_title(f"{title}: nearest-heavy distance / heavy NN ({len(allr)} Cu)"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "cu_site_geometry.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# Learned shape parameters
# blob widths vs measured 2.75 px column sigma
# chain spacings vs crystal spacings they were measured at
for nm, m in [("V1", v1), ("V2a", v2a), ("V2b", v2b)]:
    bp, cp = m.blob.params(), m.chain.params()
    a, t = count_parameters(m)
    print(f"{nm}: {t} parameters ({a} analytic)")
    print("  blob sigma px :", np.round(bp["sigma"], 2), " axis ratio:", np.round(bp["axis_ratio"], 2))
    print("  chain spacing :", np.round(cp["spacing_px"], 1), " (init", [s for s, _ in ChainBank.INIT], ")")
    print("  chain angle   :", np.round(cp["angle_deg"], 1))

summary = dict(thresholds=dict(V1=T_V1, V2a=T_V2A, V2b=T_V2B),
               synthetic_test=dict(V1=v1_test, V2a=v2a_test, V2b=v2b_test),
               best_epochs=dict(V1=v1_log["best_epoch"], V2a=v2a_log["best_epoch"], V2b=v2b_log["best_epoch"]))

(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1, default=float))
print("saved to", OUTPUT_DIR.resolve())

---
## Results summary (written automatically from the run)
This cell turns the numbers computed above into `RESULTS.md`, which the **Results at a glance** cell at the top
displays.

In [ ]:
def _f(x, n=3):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{n}f}"

L = ["# ShapeNet-YBCO — results", "",
     f"Device: {DEVICE}. Training crops {N_TRAIN}, validation {N_VAL}, test {N_TEST}, epochs {EPOCHS}; "
     f"best epochs V1 {v1_log['best_epoch']}, V2a {v2a_log['best_epoch']}, V2b {v2b_log['best_epoch']}.", "",
     "## Synthetic test set (ground truth known; 4 px matching tolerance; threshold frozen from validation)", "",
     "| Model | Class | Threshold | Precision | Recall | F1 | RMSE / px | TP | FP | FN |", "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|"]
rows_t = [("V1 heavy", "heavy", v1_test), ("V2a Cu direct", "heavy", v2a_test["heavy"]), ("V2a Cu direct", "Cu", v2a_test["Cu"]),
          ("V2b Cu residual", "Cu", v2b_test)]
for m, cl, r in rows_t:
    edge = " (grid edge)" if r["threshold"] in (min(THRESHOLDS), max(THRESHOLDS)) else ""
    L.append(f"| {m} | {cl} | {r['threshold']}{edge} | {_f(r['precision'])} | {_f(r['recall'])} | {_f(r['f1'])} | {_f(r['rmse_px'], 2)} | {r['tp']} | {r['fp']} | {r['fn']} |")
L += ["", "## Detection limit: F1 against rendered Cu/heavy brightness", "",
      f"Measured real Cu/heavy (average cells, Section A): median {_f(kept_clean.Cu_heavy.median())}, "
      f"IQR {_f(kept_clean.Cu_heavy.quantile(.25))}–{_f(kept_clean.Cu_heavy.quantile(.75))}.", "",
      "| Cu/heavy | " + " | ".join(curves) + " |", "|---:|" + "---:|" * len(curves)]
for i, r in enumerate(SWEEP_RATIOS):
    L.append(f"| {r:.2f} | " + " | ".join(_f(c[i]["f1"]) for c in curves.values()) + " |")
med = ratio_df.groupby("version")[["cu_per_heavy", "cu2_per_cu1", "site_ratio_median", "frac_site_below_0p6", "unclassified"]].median()
L += ["", "## Real images (14, no ground truth): medians over images", "",
      "Crystal reference: Cu : heavy = 1, Cu2 : Cu1 = 2; site ratio ≈ 0.70 (Cu2) / 0.80 (Cu1) from the measured average cells; 0.5 = edge midpoint.", "",
      "| Version | Cu : heavy | Cu2 : Cu1 | site ratio | fraction of Cu with site ratio < 0.6 | unclassified Cu per image |", "|---|---:|---:|---:|---:|---:|"]
for v, r in med.iterrows():
    L.append(f"| {v} | {_f(r.cu_per_heavy)} | {_f(r.cu2_per_cu1)} | {_f(r.site_ratio_median)} | {_f(r.frac_site_below_0p6)} | {r.unclassified:.0f} |")
L += ["", "Per-image values: `real_image_ratios.csv`. Images whose Cu2 : Cu1 lies outside 1.5–2.5 "
      "(a ±25% band around the crystal value, used only to list images worth inspecting):", ""]
for v in ratio_df.version.unique():
    off = ratio_df[(ratio_df.version == v) & ((ratio_df.cu2_per_cu1 < 1.5) | (ratio_df.cu2_per_cu1 > 2.5))]
    L.append(f"* {v}: " + (", ".join(f"{a[:4]} ({b:.2f})" for a, b in zip(off.image, off.cu2_per_cu1)) or "none"))
L += ["", "## Learned shape parameters", ""]
for nm, m in [("V1", v1), ("V2a", v2a), ("V2b", v2b)]:
    bp, cp = m.blob.params(), m.chain.params(); a, t = count_parameters(m)
    L.append(f"* **{nm}** ({t} parameters, {a} analytic): blob σ {np.round(bp['sigma'], 2).tolist()} px; "
             f"chain spacing {np.round(cp['spacing_px'], 1).tolist()} px (initial {[s_ for s_, _ in ChainBank.INIT]}); "
             f"chain angle {np.round(cp['angle_deg'], 1).tolist()}°")
L += ["", "Measured column σ on the real images (Calibration section): median "
      f"{_f(calib_df[calib_df.source == 'real'].sigma_px.median(), 2)} px."]
L += ["", f"## Negative control (Cu/heavy {CTRL_CU_RATIO}): empty Cu sites marked as Cu", "",
      "| Set | Model | Threshold | Empty sites | Marked | Marking rate | Recall on occupied Cu |", "|---|---|---:|---:|---:|---:|---:|"]
for _, r in control_df.iterrows():
    L.append(f"| {r['set']} | {r.model} | {r.threshold} | {r.vacant_sites} | {r.vacant_marked} | {_f(r.vacant_marked_rate)} | {_f(r.occupied_recall)} |")
RESULTS_MD = "\n".join(L)
(OUTPUT_DIR / "RESULTS.md").write_text(RESULTS_MD)
display(Markdown(RESULTS_MD))

---
## Reproducibility record and sharing

In [ ]:
def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = dict(
    created=time.strftime("%Y-%m-%d %H:%M:%S"), python=sys.version.split()[0], platform=platform.platform(),
    packages=dict(numpy=np.__version__, scipy=scipy.__version__, torch=torch.__version__, pandas=pd.__version__,
                  h5py=h5py.__version__, matplotlib=matplotlib.__version__),
    device=DEVICE, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    config=dict(CANON_PERIOD=CANON_PERIOD, ATOM_RADIUS_REL=ATOM_RADIUS_REL, CROP=CROP, AUG_PER_CROP=AUG_PER_CROP,
                AUG_SCALE_JITTER=AUG_SCALE_JITTER, AUG_ROTATION_DEG=AUG_ROTATION_DEG, AUG_GAMMA=AUG_GAMMA,
                AUG_CONTRAST=AUG_CONTRAST, AUG_NOISE_SIGMA=AUG_NOISE_SIGMA, AUG_BLUR_SIGMA_REL=AUG_BLUR_SIGMA_REL,
                MATCH_TOL=MATCH_TOL, MIN_PEAK_DIST=MIN_PEAK_DIST, THRESHOLDS=THRESHOLDS, BATCH_SIZE=BATCH_SIZE,
                LEARNING_RATE=LEARNING_RATE, EPOCHS=EPOCHS, N_TRAIN=N_TRAIN, N_VAL=N_VAL, N_TEST=N_TEST,
                SWEEP_RATIOS=SWEEP_RATIOS, N_PER_RATIO=N_PER_RATIO, SEEDS=SEEDS, DIST_SCALE=DIST_SCALE, DROP_IDS=DROP_IDS,
                N_CTRL=N_CTRL, CTRL_CU_RATIO=CTRL_CU_RATIO, VACANT_CODE=VACANT_CODE),
    generator=GEN,
    thresholds=dict(V1=T_V1, V2a=T_V2A, V2b=T_V2B),
    images_used={f.name: _sha256(f) for f in ybco_files if f.stem[:4] not in DROP_IDS},
    images_dropped={f.name: _sha256(f) for f in ybco_files if f.stem[:4] in DROP_IDS},
    scans=scan_df.reset_index().astype({"native_shape": str, "canon_shape": str}).to_dict(orient="records"))
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str))
zip_path = shutil.make_archive("shapenet_ybco_outputs", "zip", OUTPUT_DIR)
print("wrote", OUTPUT_DIR / "manifest.json", "and", zip_path)
share_dir = Path("/content/drive/MyDrive/ShapeNet_YBCO_Day25")
if Path("/content/drive/MyDrive").exists():
    share_dir.mkdir(parents=True, exist_ok=True); shutil.copy(zip_path, share_dir)
    print("copied to", share_dir, "- share this folder and the notebook with the team")
else:
    print("Drive not mounted: download", zip_path, "manually to share it")